In [1]:
import sys
sys.path.insert(0, '../')

import pandas as pd

from automed import *
from IPython.display import display, Markdown as IMarkdown
from rich.console import Console
from rich.markdown import Markdown

[11:17:14] cuDF not found: falling back to standalone pandas.

In [2]:
titanic = pd.read_csv('../perf_logger/tests_data/titanic.csv', delimiter=';')

In [3]:
autom = AutoMed()

print(autom.default_pipeline())
print(autom.json_pipeline())

None
{'step': 'MetaOrderedStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'MetaStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActOnehot', 'name': 'One hot encoding categorical features', 'description': 'Step description...', 'configuration': {}, 'children': []}, {'step': 'ActMeanColumn', 'name': 'Fill missing values with mean', 'description': 'Fills missing values with the mean of non-missing values when the proportion of empty rows is lower than {empty_threshold}.', 'configuration': {'empty_threshold': {'description': 'Column with less or equal proportion of empty row will                    be fill with mean value. 1 will always fill void values', 'default': 0.5, 'value': 0.5}}, 'children': []}, {'step': 'ActDropTextualColumn', 'name': 'Drop textual columns', 'description': 'Drop textual columns.', 'configuration': {}, 'children': []}, {'step': 'ActSplitDate', 'na

In [4]:
pipeline = {
    'step': 'MetaOrderedStep',
    'children': [
        {
            'step': 'ActDropNumericalColumn',
            'configuration': {
                'empty_threshold': { 'value': 0.1 },
            }
        },
        {
            'step': 'MetaStep',
            'tag': 'cleaning',
        },
        {
            'step': 'MetaStep',
            'tag': 'features_selection',
        },
        {
            'step': 'WrapKFold',
            'children': [{
                'step': 'ActKNN',
            }]
        },
    ]
}

autom.load_pipeline(pipeline)
print(autom.json_pipeline())


{'step': 'MetaOrderedStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActDropNumericalColumn', 'name': 'Drop numerical columns', 'description': 'Drop numerical columns where the proportion of empty rows\n        in the dataset is higher than {empty_threshold}.', 'configuration': {'empty_threshold': {'description': 'Column with more or equal proportion of empty row                     will dropped. 1 will drop all columns', 'default': 0.5, 'value': 0.1}}, 'children': []}, {'step': 'MetaStep', 'name': 'MetaStep', 'description': 'Step description...', 'configuration': {}, 'children': [{'step': 'ActOnehot', 'name': 'One hot encoding categorical features', 'description': 'Step description...', 'configuration': {}, 'children': []}, {'step': 'ActMeanColumn', 'name': 'Fill missing values with mean', 'description': 'Fills missing values with the mean of non-missing values when the proportion of empty rows is lower than {empty_threshold

In [5]:
results = autom.fit(
    titanic.drop('label', axis=1).copy(),
    titanic[['label']]).copy()

Output()

[11:17:15] running step: MetaOrderedStep (steps=MetaStep,ActDropNumericalColumn,WrapKFold)

[11:17:19] running step: MetaStep (steps=ActRemoveHighCorrelatedColumn)

[11:17:28] running step: WrapKFold (step=ActKNN, folds=5, stratify=True)

c:\Users\0869778\dev\automl\.venv\Lib\site-packages\sklearn\neighbors\_classification.py:233: 
DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to 
(n_samples,), for example using ravel().
  return self._fit(X, y)

c:\Users\0869778\dev\automl\.venv\Lib\site-packages\sklearn\neighbors\_classification.py:233: 
DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to 
(n_samples,), for example using ravel().
  return self._fit(X, y)

c:\Users\0869778\dev\automl\.venv\Lib\site-packages\sklearn\neighbors\_classification.py:233: 
DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to 
(n_samples,), for example using ravel().
  return self._fit(X, y)

c:\Users\0869778\dev\automl\.venv\Lib\site-packages\sklearn\neighbors\_classification.py:233: 
DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to 
(n_samples,), for example using ravel().
  return self._fit(X, y)

c:\Users\0869778\dev\automl\.venv\Lib\site-packages\sklearn\neighbors\_classification.py:233: 
DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to 
(n_samples,), for example using ravel().
  return self._fit(X, y)

c:\Users\0869778\dev\automl\.venv\Lib\site-packages\sklearn\neighbors\_classification.py:233: 
DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to 
(n_samples,), for example using ravel().
  return self._fit(X, y)

In [6]:
print(results[0].pipeline.model)
print(results[0].pipeline.steps)

console = Console()

for step in results[0].pipeline.explanations:
    md = step.to_markdown()
    if md:
        # console.print(Markdown(md))
        display(IMarkdown(md))

pm = results[0].pipeline.pickle()

Learn : KNN
[('Drop numerical columns', <automed.actionables.cleaning.act_drop_numerical_column.ActDropNumericalColumn object at 0x0000020565DC1B10>), ('Fill missing values with mean', <automed.actionables.cleaning.act_mean_column.ActMeanColumn object at 0x0000020565DF5CD0>), ('One hot encoding categorical features', <automed.actionables.cleaning.act_onehot.ActOnehot object at 0x0000020565DF5C50>), ('Transform string column to date', <automed.actionables.cleaning.act_split_date.ActSplitDate object at 0x0000020565DF5E10>), ('TF-IDF', <automed.actionables.cleaning.act_tf_idf.ActTfIdf object at 0x0000020565DF6150>), ('Drop textual columns', <automed.actionables.cleaning.act_drop_textual_column.ActDropTextualColumn object at 0x0000020565DF5D50>), ('Drop numerical columns', <automed.actionables.cleaning.act_drop_numerical_column.ActDropNumericalColumn object at 0x0000020565DF5F10>), ('Drop date columns', <automed.actionables.cleaning.act_drop_date_column.ActDropDateColumn object at 0x000002


## Drop numerical columns
**Drop numerical columns where the proportion of empty rows
        in the dataset is higher than 0.1.**


### Configuration
| Name | Description | Value |
| ---- | ----------- | ----- |
| **empty_threshold** | Column with more or equal proportion of empty row                     will dropped. 1 will drop all columns | 0.1 |



### Processings
 - Dropped column **`Age`** because **177** values out of **891** (**19.87%**) are empty.

            


## One hot encoding categorical features
**Step description...**




### Processings
 - Encoded categorical column **`Sex`** into **2** new columns.
 - Encoded categorical column **`Embarked`** into **4** new columns.

            


## TF-IDF
**Step description...**




### Processings
 - Encoded text column **`Name`** into **1509** new columns.
 - Encoded text column **`Ticket`** into **695** new columns.
 - Encoded text column **`Cabin`** into **158** new columns.

            


## Remove High Correlated Column
**Remove columns which correlation with other columns is higher than 0.9.**


### Configuration
| Name | Description | Value |
| ---- | ----------- | ----- |
| **threshold** | If two columns is correlated over this value, only one                     will be kept | 0.9 |



### Processings
 - Dropped column **`Sex_male`** because it was too correlated with **`Sex_female`**.
 - Dropped column **`Name_amanda`** because it was too correlated with **`Name_adolfina`**.
 - Dropped column **`Name_antino`** because it was too correlated with **`Name_aijo`**.
 - Dropped column **`Name_asim`** because it was too correlated with **`Name_adola`**.
 - Dropped column **`Name_aurora`** because it was too correlated with **`Name_adelia`**.
 - Dropped column **`Name_banoura`** because it was too correlated with **`Name_ayoub`**.
 - Dropped column **`Name_barah`** because it was too correlated with **`Name_assi`**.
 - Dropped column **`Name_barkworth`** because it was too correlated with **`Name_algernon`**.
 - Dropped column **`Name_bazzani`** because it was too correlated with **`Name_albina`**.
 - Dropped column **`Name_berthe`** because it was too correlated with **`Name_antonine`**.
 - Dropped column **`Name_blyler`** because it was too correlated with **`Name_billiard`**.
 - Dropped column **`Name_bratthammer`** because it was too correlated with **`Name_bernt`**.
 - Dropped column **`Name_butt`** because it was too correlated with **`Name_archibald`**.
 - Dropped column **`Name_cassem`** because it was too correlated with **`Name_albimona`**.
 - Dropped column **`Name_cerin`** because it was too correlated with **`Name_balkic`**.
 - Dropped column **`Name_chip`** because it was too correlated with **`Name_chang`**.
 - Dropped column **`Name_christine`** because it was too correlated with **`Name_carla`**.
 - Dropped column **`Name_chronopoulos`** because it was too correlated with **`Name_apostolos`**.
 - Dropped column **`Name_clear`** because it was too correlated with **`Name_cameron`**.
 - Dropped column **`Name_cornelia`** because it was too correlated with **`Name_alma`**.
 - Dropped column **`Name_cumings`** because it was too correlated with **`Name_briggs`**.
 - Dropped column **`Name_dai`** because it was too correlated with **`Name_bowen`**.
 - Dropped column **`Name_dakic`** because it was too correlated with **`Name_branko`**.
 - Dropped column **`Name_dale`** because it was too correlated with **`Name_appleton`**.
 - Dropped column **`Name_davids`** because it was too correlated with **`Name_byles`**.
 - Dropped column **`Name_del`** because it was too correlated with **`Name_carlo`**.
 - Dropped column **`Name_delaudeniere`** because it was too correlated with **`Name_chaput`**.
 - Dropped column **`Name_dick`** because it was too correlated with **`Name_adrian`**.
 - Dropped column **`Name_dickinson`** because it was too correlated with **`Name_bishop`**.
 - Dropped column **`Name_domingos`** because it was too correlated with **`Name_coelho`**.
 - Dropped column **`Name_drake`** because it was too correlated with **`Name_cardeza`**.
 - Dropped column **`Name_duran`** because it was too correlated with **`Name_asuncion`**.
 - Dropped column **`Name_dyer`** because it was too correlated with **`Name_countess`**.
 - Dropped column **`Name_edmund`** because it was too correlated with **`Name_cosmo`**.
 - Dropped column **`Name_edwards`** because it was too correlated with **`Name_countess`**, **`Name_dyer`**.
 - Dropped column **`Name_edwart`** because it was too correlated with **`Name_dahl`**.
 - Dropped column **`Name_edwina`** because it was too correlated with **`Name_celia`**.
 - Dropped column **`Name_elin`** because it was too correlated with **`Name_dolck`**.
 - Dropped column **`Name_emir`** because it was too correlated with **`Name_chehab`**.
 - Dropped column **`Name_endres`** because it was too correlated with **`Name_caroline`**.
 - Dropped column **`Name_engelhart`** because it was too correlated with **`Name_cornelius`**.
 - Dropped column **`Name_esther`** because it was too correlated with **`Name_bloomfield`**.
 - Dropped column **`Name_farred`** because it was too correlated with **`Name_chehab`**, **`Name_emir`**.
 - Dropped column **`Name_fenton`** because it was too correlated with **`Name_butler`**.
 - Dropped column **`Name_fernandeo`** because it was too correlated with **`Name_coelho`**, **`Name_domingos`**.
 - Dropped column **`Name_finck`** because it was too correlated with **`Name_davison`**.
 - Dropped column **`Name_fletcher`** because it was too correlated with **`Name_fellows`**.
 - Dropped column **`Name_floyd`** because it was too correlated with **`Name_eitemiller`**.
 - Dropped column **`Name_foo`** because it was too correlated with **`Name_choong`**.
 - Dropped column **`Name_force`** because it was too correlated with **`Name_astor`**.
 - Dropped column **`Name_francesco`** because it was too correlated with **`Name_celotti`**.
 - Dropped column **`Name_francisco`** because it was too correlated with **`Name_carrau`**.
 - Dropped column **`Name_frost`** because it was too correlated with **`Name_archie`**.
 - Dropped column **`Name_fuller`** because it was too correlated with **`Name_chaffee`**.
 - Dropped column **`Name_funk`** because it was too correlated with **`Name_clemmer`**.
 - Dropped column **`Name_gates`** because it was too correlated with **`Name_alden`**.
 - Dropped column **`Name_gerda`** because it was too correlated with **`Name_dahlberg`**.
 - Dropped column **`Name_gifford`** because it was too correlated with **`Name_capt`**.
 - Dropped column **`Name_gilbert`** because it was too correlated with **`Name_danbom`**.
 - Dropped column **`Name_gilinski`** because it was too correlated with **`Name_eliezer`**.
 - Dropped column **`Name_glynn`** because it was too correlated with **`Name_agatha`**.
 - Dropped column **`Name_goncalves`** because it was too correlated with **`Name_estanslas`**.
 - Dropped column **`Name_gordon`** because it was too correlated with **`Name_duff`**.
 - Dropped column **`Name_gottfrid`** because it was too correlated with **`Name_bryhl`**.
 - Dropped column **`Name_grabowska`** because it was too correlated with **`Name_edwiga`**.
 - Dropped column **`Name_grace`** because it was too correlated with **`Name_charity`**.
 - Dropped column **`Name_gronnestad`** because it was too correlated with **`Name_danielsen`**.
 - Dropped column **`Name_guentcho`** because it was too correlated with **`Name_bostandyeff`**.
 - Dropped column **`Name_gurshon`** because it was too correlated with **`Name_cohen`**.
 - Dropped column **`Name_gus`** because it was too correlated with **`Name_cohen`**, **`Name_gurshon`**.
 - Dropped column **`Name_haas`** because it was too correlated with **`Name_aloisia`**.
 - Dropped column **`Name_hakan`** because it was too correlated with **`Name_bjornstrom`**.
 - Dropped column **`Name_halim`** because it was too correlated with **`Name_gonios`**.
 - Dropped column **`Name_hanne`** because it was too correlated with **`Name_darwis`**.
 - Dropped column **`Name_harder`** because it was too correlated with **`Name_achilles`**.
 - Dropped column **`Name_hassab`** because it was too correlated with **`Name_hammad`**.
 - Dropped column **`Name_hastings`** because it was too correlated with **`Name_ennis`**.
 - Dropped column **`Name_hatfield`** because it was too correlated with **`Name_cribb`**.
 - Dropped column **`Name_heath`** because it was too correlated with **`Name_futrelle`**.
 - Dropped column **`Name_henriette`** because it was too correlated with **`Name_harbeck`**.
 - Dropped column **`Name_herbert`** because it was too correlated with **`Name_chaffee`**, **`Name_fuller`**.
 - Dropped column **`Name_hirvonen`** because it was too correlated with **`Name_hildur`**.
 - Dropped column **`Name_hitchcock`** because it was too correlated with **`Name_dennick`**.
 - Dropped column **`Name_hoef`** because it was too correlated with **`Name_der`**.
 - Dropped column **`Name_holm`** because it was too correlated with **`Name_fredrik`**.
 - Dropped column **`Name_homer`** because it was too correlated with **`Name_haven`**.
 - Dropped column **`Name_honkanen`** because it was too correlated with **`Name_eliina`**.
 - Dropped column **`Name_hood`** because it was too correlated with **`Name_ambrose`**.
 - Dropped column **`Name_houssein`** because it was too correlated with **`Name_hassan`**.
 - Dropped column **`Name_howell`** because it was too correlated with **`Name_behr`**.
 - Dropped column **`Name_hubert`** because it was too correlated with **`Name_fox`**.
 - Dropped column **`Name_hulda`** because it was too correlated with **`Name_adolfina`**, **`Name_amanda`**.
 - Dropped column **`Name_icard`** because it was too correlated with **`Name_amelie`**.
 - Dropped column **`Name_ignjac`** because it was too correlated with **`Name_hendekovic`**.
 - Dropped column **`Name_iisakki`** because it was too correlated with **`Name_aijo`**, **`Name_antino`**.
 - Dropped column **`Name_ileen`** because it was too correlated with **`Name_eleanor`**.
 - Dropped column **`Name_ilmari`** because it was too correlated with **`Name_alhomaki`**.
 - Dropped column **`Name_ingeborg`** because it was too correlated with **`Name_constanzia`**.
 - Dropped column **`Name_iris`** because it was too correlated with **`Name_ebba`**.
 - Dropped column **`Name_irwin`** because it was too correlated with **`Name_irving`**.
 - Dropped column **`Name_jackson`** because it was too correlated with **`Name_brewe`**.
 - Dropped column **`Name_jan`** because it was too correlated with **`Name_baptist`**.
 - Dropped column **`Name_jarvis`** because it was too correlated with **`Name_denzil`**.
 - Dropped column **`Name_jenkin`** because it was too correlated with **`Name_curnow`**.
 - Dropped column **`Name_jennings`** because it was too correlated with **`Name_gregg`**.
 - Dropped column **`Name_jeremiah`** because it was too correlated with **`Name_burke`**.
 - Dropped column **`Name_jeso`** because it was too correlated with **`Name_culumovic`**.
 - Dropped column **`Name_johannesen`** because it was too correlated with **`Name_bernt`**, **`Name_bratthammer`**.
 - Dropped column **`Name_jonas`** because it was too correlated with **`Name_fahlstrom`**.
 - Dropped column **`Name_jose`** because it was too correlated with **`Name_jardin`**.
 - Dropped column **`Name_josef`** because it was too correlated with **`Name_franchi`**.
 - Dropped column **`Name_josefina`** because it was too correlated with **`Name_helmina`**.
 - Dropped column **`Name_jovan`** because it was too correlated with **`Name_dimic`**.
 - Dropped column **`Name_jozef`** because it was too correlated with **`Name_drazenoic`**.
 - Dropped column **`Name_julia`** because it was too correlated with **`Name_bone`**.
 - Dropped column **`Name_juliet`** because it was too correlated with **`Name_cummins`**.
 - Dropped column **`Name_julius`** because it was too correlated with **`Name_emelia`**.
 - Dropped column **`Name_kallio`** because it was too correlated with **`Name_erland`**.
 - Dropped column **`Name_kalvik`** because it was too correlated with **`Name_halvorsen`**.
 - Dropped column **`Name_kanio`** because it was too correlated with **`Name_ivanoff`**.
 - Dropped column **`Name_karolina`** because it was too correlated with **`Name_bystrom`**.
 - Dropped column **`Name_kassem`** because it was too correlated with **`Name_fared`**.
 - Dropped column **`Name_kimball`** because it was too correlated with **`Name_edwin`**.
 - Dropped column **`Name_kingcome`** because it was too correlated with **`Name_hewlett`**.
 - Dropped column **`Name_klas`** because it was too correlated with **`Name_albin`**.
 - Dropped column **`Name_klasen`** because it was too correlated with **`Name_albin`**, **`Name_klas`**.
 - Dropped column **`Name_kristensen`** because it was too correlated with **`Name_givard`**.
 - Dropped column **`Name_kurt`** because it was too correlated with **`Name_bryhl`**, **`Name_gottfrid`**.
 - Dropped column **`Name_kvillner`** because it was too correlated with **`Name_johannesson`**.
 - Dropped column **`Name_lady`** because it was too correlated with **`Name_christiana`**.
 - Dropped column **`Name_lafargue`** because it was too correlated with **`Name_juliette`**.
 - Dropped column **`Name_laina`** because it was too correlated with **`Name_heikkinen`**.
 - Dropped column **`Name_laitinen`** because it was too correlated with **`Name_kristina`**.
 - Dropped column **`Name_laleff`** because it was too correlated with **`Name_kristo`**.
 - Dropped column **`Name_lambert`** because it was too correlated with **`Name_fellows`**, **`Name_fletcher`**.
 - Dropped column **`Name_lamson`** because it was too correlated with **`Name_appleton`**, **`Name_dale`**.
 - Dropped column **`Name_landergren`** because it was too correlated with **`Name_adelia`**, **`Name_aurora`**.
 - Dropped column **`Name_lang`** because it was too correlated with **`Name_fang`**.
 - Dropped column **`Name_laura`** because it was too correlated with **`Name_francatelli`**.
 - Dropped column **`Name_laury`** because it was too correlated with **`Name_charity`**, **`Name_grace`**.
 - Dropped column **`Name_laventall`** because it was too correlated with **`Name_foreman`**.
 - Dropped column **`Name_leader`** because it was too correlated with **`Name_farnham`**.
 - Dropped column **`Name_leah`** because it was too correlated with **`Name_aks`**.
 - Dropped column **`Name_leeni`** because it was too correlated with **`Name_fahim`**.
 - Dropped column **`Name_leon`** because it was too correlated with **`Name_hampe`**.
 - Dropped column **`Name_leontine`** because it was too correlated with **`Name_aubart`**.
 - Dropped column **`Name_leopold`** because it was too correlated with **`Name_francoise`**.
 - Dropped column **`Name_lesurer`** because it was too correlated with **`Name_gustave`**.
 - Dropped column **`Name_lewy`** because it was too correlated with **`Name_ervin`**.
 - Dropped column **`Name_lievens`** because it was too correlated with **`Name_aime`**.
 - Dropped column **`Name_lindahl`** because it was too correlated with **`Name_agda`**.
 - Dropped column **`Name_lines`** because it was too correlated with **`Name_conover`**.
 - Dropped column **`Name_linus`** because it was too correlated with **`Name_eklund`**.
 - Dropped column **`Name_lishin`** because it was too correlated with **`Name_harmer`**.
 - Dropped column **`Name_liudevit`** because it was too correlated with **`Name_cor`**.
 - Dropped column **`Name_lizzie`** because it was too correlated with **`Name_faunthorpe`**.
 - Dropped column **`Name_long`** because it was too correlated with **`Name_clyde`**.
 - Dropped column **`Name_longley`** because it was too correlated with **`Name_fiske`**.
 - Dropped column **`Name_louch`** because it was too correlated with **`Name_adelaide`**.
 - Dropped column **`Name_lucille`** because it was too correlated with **`Name_christiana`**, **`Name_lady`**.
 - Dropped column **`Name_luise`** because it was too correlated with **`Name_heilmann`**.
 - Dropped column **`Name_lulu`** because it was too correlated with **`Name_drew`**.
 - Dropped column **`Name_lurette`** because it was too correlated with **`Name_elise`**.
 - Dropped column **`Name_lyyli`** because it was too correlated with **`Name_karoliina`**.
 - Dropped column **`Name_madill`** because it was too correlated with **`Name_georgette`**.
 - Dropped column **`Name_madsen`** because it was too correlated with **`Name_fridtjof`**.
 - Dropped column **`Name_mae`** because it was too correlated with **`Name_harbaugh`**.
 - Dropped column **`Name_maenpaa`** because it was too correlated with **`Name_alexanteri`**.
 - Dropped column **`Name_manca`** because it was too correlated with **`Name_karun`**.
 - Dropped column **`Name_manley`** because it was too correlated with **`Name_atkinson`**.
 - Dropped column **`Name_margareth`** because it was too correlated with **`Name_mannion`**.
 - Dropped column **`Name_margaretta`** because it was too correlated with **`Name_corning`**.
 - Dropped column **`Name_mari`** because it was too correlated with **`Name_aina`**.
 - Dropped column **`Name_marian`** because it was too correlated with **`Name_longstreth`**.
 - Dropped column **`Name_markoff`** because it was too correlated with **`Name_marin`**.
 - Dropped column **`Name_markun`** because it was too correlated with **`Name_johann`**.
 - Dropped column **`Name_marthe`** because it was too correlated with **`Name_jerwan`**.
 - Dropped column **`Name_martinez`** because it was too correlated with **`Name_cardeza`**, **`Name_drake`**.
 - Dropped column **`Name_masabumi`** because it was too correlated with **`Name_hosono`**.
 - Dropped column **`Name_masselmani`** because it was too correlated with **`Name_fatima`**.
 - Dropped column **`Name_max`** because it was too correlated with **`Name_maeglin`**.
 - Dropped column **`Name_mayne`** because it was too correlated with **`Name_antonine`**, **`Name_berthe`**.
 - Dropped column **`Name_mcdermott`** because it was too correlated with **`Name_brigdet`**.
 - Dropped column **`Name_melville`** because it was too correlated with **`Name_gregg`**, **`Name_jennings`**.
 - Dropped column **`Name_meo`** because it was too correlated with **`Name_alfonzo`**.
 - Dropped column **`Name_messemaeker`** because it was too correlated with **`Name_guillaume`**.
 - Dropped column **`Name_milan`** because it was too correlated with **`Name_karaic`**.
 - Dropped column **`Name_milley`** because it was too correlated with **`Name_lemore`**.
 - Dropped column **`Name_milne`** because it was too correlated with **`Name_inglis`**.
 - Dropped column **`Name_milton`** because it was too correlated with **`Name_clyde`**, **`Name_long`**.
 - Dropped column **`Name_mito`** because it was too correlated with **`Name_mitkoff`**.
 - Dropped column **`Name_mitto`** because it was too correlated with **`Name_denkoff`**.
 - Dropped column **`Name_mme`** because it was too correlated with **`Name_aubart`**, **`Name_leontine`**.
 - Dropped column **`Name_mockler`** because it was too correlated with **`Name_ellie`**.
 - Dropped column **`Name_mohamed`** because it was too correlated with **`Name_badt`**.
 - Dropped column **`Name_molson`** because it was too correlated with **`Name_markland`**.
 - Dropped column **`Name_monsen`** because it was too correlated with **`Name_birkeland`**.
 - Dropped column **`Name_montvila`** because it was too correlated with **`Name_juozas`**.
 - Dropped column **`Name_more`** because it was too correlated with **`Name_asuncion`**, **`Name_duran`**.
 - Dropped column **`Name_morris`** because it was too correlated with **`Name_longstreth`**, **`Name_marian`**.
 - Dropped column **`Name_moses`** because it was too correlated with **`Name_aaron`**.
 - Dropped column **`Name_moussa`** because it was too correlated with **`Name_mantoura`**.
 - Dropped column **`Name_moutal`** because it was too correlated with **`Name_haim`**.
 - Dropped column **`Name_ms`** because it was too correlated with **`Name_encarnacion`**.
 - Dropped column **`Name_myhrman`** because it was too correlated with **`Name_fabian`**.
 - Dropped column **`Name_myna`** because it was too correlated with **`Name_haxtun`**.
 - Dropped column **`Name_najib`** because it was too correlated with **`Name_kiamie`**.
 - Dropped column **`Name_nankoff`** because it was too correlated with **`Name_minko`**.
 - Dropped column **`Name_nassef`** because it was too correlated with **`Name_albimona`**, **`Name_cassem`**.
 - Dropped column **`Name_needs`** because it was too correlated with **`Name_eliza`**.
 - Dropped column **`Name_nenkoff`** because it was too correlated with **`Name_christo`**.
 - Dropped column **`Name_nestor`** because it was too correlated with **`Name_cyriel`**.
 - Dropped column **`Name_neto`** because it was too correlated with **`Name_jardin`**, **`Name_jose`**.
 - Dropped column **`Name_neville`** because it was too correlated with **`Name_eden`**.
 - Dropped column **`Name_nicholas`** because it was too correlated with **`Name_nasser`**.
 - Dropped column **`Name_nicolai`** because it was too correlated with **`Name_humblen`**.
 - Dropped column **`Name_nielsine`** because it was too correlated with **`Name_carla`**, **`Name_christine`**.
 - Dropped column **`Name_nikola`** because it was too correlated with **`Name_lulic`**.
 - Dropped column **`Name_nikolai`** because it was too correlated with **`Name_erland`**, **`Name_kallio`**.
 - Dropped column **`Name_nilsson`** because it was too correlated with **`Name_helmina`**, **`Name_josefina`**.
 - Dropped column **`Name_nirva`** because it was too correlated with **`Name_aijo`**, **`Name_antino`**, **`Name_iisakki`**.
 - Dropped column **`Name_noel`** because it was too correlated with **`Name_countess`**, **`Name_dyer`**, **`Name_edwards`**.
 - Dropped column **`Name_norah`** because it was too correlated with **`Name_leary`**.
 - Dropped column **`Name_nosworthy`** because it was too correlated with **`Name_cater`**.
 - Dropped column **`Name_novel`** because it was too correlated with **`Name_mansouer`**.
 - Dropped column **`Name_oakley`** because it was too correlated with **`Name_corning`**, **`Name_margaretta`**.
 - Dropped column **`Name_oberst`** because it was too correlated with **`Name_blumer`**.
 - Dropped column **`Name_of`** because it was too correlated with **`Name_countess`**, **`Name_dyer`**, **`Name_edwards`**, **`Name_noel`**.
 - Dropped column **`Name_ogden`** because it was too correlated with **`Name_meanwell`**.
 - Dropped column **`Name_olaf`** because it was too correlated with **`Name_elon`**.
 - Dropped column **`Name_olai`** because it was too correlated with **`Name_ingvald`**.
 - Dropped column **`Name_oliver`** because it was too correlated with **`Name_fabian`**, **`Name_myhrman`**.
 - Dropped column **`Name_osen`** because it was too correlated with **`Name_elon`**, **`Name_olaf`**.
 - Dropped column **`Name_osman`** because it was too correlated with **`Name_mara`**.
 - Dropped column **`Name_ostby`** because it was too correlated with **`Name_cornelius`**, **`Name_engelhart`**.
 - Dropped column **`Name_padro`** because it was too correlated with **`Name_manent`**.
 - Dropped column **`Name_parr`** because it was too correlated with **`Name_marsh`**.
 - Dropped column **`Name_partner`** because it was too correlated with **`Name_austen`**.
 - Dropped column **`Name_paul`** because it was too correlated with **`Name_andreasson`**.
 - Dropped column **`Name_paula`** because it was too correlated with **`Name_govaert`**.
 - Dropped column **`Name_pauline`** because it was too correlated with **`Name_aubart`**, **`Name_leontine`**, **`Name_mme`**.
 - Dropped column **`Name_paust`** because it was too correlated with **`Name_knud`**.
 - Dropped column **`Name_pede`** because it was too correlated with **`Name_francoise`**, **`Name_leopold`**.
 - Dropped column **`Name_pehr`** because it was too correlated with **`Name_fabian`**, **`Name_myhrman`**, **`Name_oliver`**.
 - Dropped column **`Name_pekka`** because it was too correlated with **`Name_hakkarainen`**.
 - Dropped column **`Name_penasco`** because it was too correlated with **`Name_castellana`**.
 - Dropped column **`Name_penko`** because it was too correlated with **`Name_naidenoff`**.
 - Dropped column **`Name_pennington`** because it was too correlated with **`Name_calderhead`**.
 - Dropped column **`Name_pentcho`** because it was too correlated with **`Name_pastcho`**.
 - Dropped column **`Name_percy`** because it was too correlated with **`Name_bailey`**.
 - Dropped column **`Name_perez`** because it was too correlated with **`Name_josefa`**.
 - Dropped column **`Name_persdotter`** because it was too correlated with **`Name_ahlin`**.
 - Dropped column **`Name_pettersson`** because it was too correlated with **`Name_natalia`**.
 - Dropped column **`Name_philemon`** because it was too correlated with **`Name_melkebeke`**.
 - Dropped column **`Name_philippe`** because it was too correlated with **`Name_lemercier`**.
 - Dropped column **`Name_phoebe`** because it was too correlated with **`Name_harknett`**.
 - Dropped column **`Name_pickard`** because it was too correlated with **`Name_berk`**.
 - Dropped column **`Name_pierre`** because it was too correlated with **`Name_marechal`**.
 - Dropped column **`Name_pieta`** because it was too correlated with **`Name_ilmakangas`**.
 - Dropped column **`Name_pietari`** because it was too correlated with **`Name_hakkarainen`**, **`Name_pekka`**.
 - Dropped column **`Name_polk`** because it was too correlated with **`Name_lucile`**.
 - Dropped column **`Name_pomeroy`** because it was too correlated with **`Name_colley`**.
 - Dropped column **`Name_porter`** because it was too correlated with **`Name_chamberlain`**.
 - Dropped column **`Name_potter`** because it was too correlated with **`Name_alexenia`**.
 - Dropped column **`Name_price`** because it was too correlated with **`Name_hodges`**.
 - Dropped column **`Name_quincy`** because it was too correlated with **`Name_clifford`**.
 - Dropped column **`Name_qurban`** because it was too correlated with **`Name_latifa`**.
 - Dropped column **`Name_rachel`** because it was too correlated with **`Name_julie`**.
 - Dropped column **`Name_rahamin`** because it was too correlated with **`Name_haim`**, **`Name_moutal`**.
 - Dropped column **`Name_ramell`** because it was too correlated with **`Name_nye`**.
 - Dropped column **`Name_ramon`** because it was too correlated with **`Name_artagaveytia`**.
 - Dropped column **`Name_rebecca`** because it was too correlated with **`Name_compton`**.
 - Dropped column **`Name_reiersen`** because it was too correlated with **`Name_konrad`**.
 - Dropped column **`Name_reuchlin`** because it was too correlated with **`Name_jonkheer`**.
 - Dropped column **`Name_reynaldo`** because it was too correlated with **`Name_encarnacion`**, **`Name_ms`**.
 - Dropped column **`Name_ridsdale`** because it was too correlated with **`Name_lucy`**.
 - Dropped column **`Name_risien`** because it was too correlated with **`Name_beard`**.
 - Dropped column **`Name_ristiu`** because it was too correlated with **`Name_dantcheff`**.
 - Dropped column **`Name_roberta`** because it was too correlated with **`Name_maioni`**.
 - Dropped column **`Name_robins`** because it was too correlated with **`Name_charity`**, **`Name_grace`**, **`Name_laury`**.
 - Dropped column **`Name_rojj`** because it was too correlated with **`Name_felix`**.
 - Dropped column **`Name_rolmane`** because it was too correlated with **`Name_hallace`**.
 - Dropped column **`Name_romaine`** because it was too correlated with **`Name_hallace`**, **`Name_rolmane`**.
 - Dropped column **`Name_rommetvedt`** because it was too correlated with **`Name_knud`**, **`Name_paust`**.
 - Dropped column **`Name_roscoe`** because it was too correlated with **`Name_rood`**.
 - Dropped column **`Name_rosen`** because it was too correlated with **`Name_aks`**, **`Name_leah`**.
 - Dropped column **`Name_rothes`** because it was too correlated with **`Name_countess`**, **`Name_dyer`**, **`Name_edwards`**, **`Name_noel`**, **`Name_of`**.
 - Dropped column **`Name_rothschild`** because it was too correlated with **`Name_barrett`**.
 - Dropped column **`Name_roussel`** because it was too correlated with **`Name_byles`**, **`Name_davids`**.
 - Dropped column **`Name_rowan`** because it was too correlated with **`Name_morrow`**.
 - Dropped column **`Name_rowley`** because it was too correlated with **`Name_meek`**.
 - Dropped column **`Name_ruby`** because it was too correlated with **`Name_robina`**.
 - Dropped column **`Name_rudolf`** because it was too correlated with **`Name_alhomaki`**, **`Name_ilmari`**.
 - Dropped column **`Name_rut`** because it was too correlated with **`Name_marguerite`**.
 - Dropped column **`Name_saalfeld`** because it was too correlated with **`Name_adolphe`**.
 - Dropped column **`Name_sadlier`** because it was too correlated with **`Name_matthew`**.
 - Dropped column **`Name_saks`** because it was too correlated with **`Name_leila`**.
 - Dropped column **`Name_salkjelsvik`** because it was too correlated with **`Name_kristine`**.
 - Dropped column **`Name_sam`** because it was too correlated with **`Name_aks`**, **`Name_leah`**, **`Name_rosen`**.
 - Dropped column **`Name_sante`** because it was too correlated with **`Name_ringhini`**.
 - Dropped column **`Name_sara`** because it was too correlated with **`Name_compton`**, **`Name_rebecca`**.
 - Dropped column **`Name_sarkis`** because it was too correlated with **`Name_lahoud`**.
 - Dropped column **`Name_satode`** because it was too correlated with **`Name_castellana`**, **`Name_penasco`**.
 - Dropped column **`Name_scott`** because it was too correlated with **`Name_mcmillan`**.
 - Dropped column **`Name_sebastiano`** because it was too correlated with **`Name_carlo`**, **`Name_del`**.
 - Dropped column **`Name_seward`** because it was too correlated with **`Name_kimber`**.
 - Dropped column **`Name_shadrach`** because it was too correlated with **`Name_gale`**.
 - Dropped column **`Name_shawah`** because it was too correlated with **`Name_ibrahim`**.
 - Dropped column **`Name_sheerlinck`** because it was too correlated with **`Name_baptist`**, **`Name_jan`**.
 - Dropped column **`Name_shelley`** because it was too correlated with **`Name_imanita`**.
 - Dropped column **`Name_sigurd`** because it was too correlated with **`Name_moen`**.
 - Dropped column **`Name_silven`** because it was too correlated with **`Name_karoliina`**, **`Name_lyyli`**.
 - Dropped column **`Name_silvey`** because it was too correlated with **`Name_baird`**.
 - Dropped column **`Name_simon`** because it was too correlated with **`Name_maisner`**.
 - Dropped column **`Name_simonius`** because it was too correlated with **`Name_blumer`**, **`Name_oberst`**.
 - Dropped column **`Name_simonne`** because it was too correlated with **`Name_andree`**.
 - Dropped column **`Name_sinai`** because it was too correlated with **`Name_kantor`**.
 - Dropped column **`Name_sir`** because it was too correlated with **`Name_cosmo`**, **`Name_edmund`**.
 - Dropped column **`Name_sirayanian`** because it was too correlated with **`Name_orsen`**.
 - Dropped column **`Name_sivic`** because it was too correlated with **`Name_husein`**.
 - Dropped column **`Name_slabenoff`** because it was too correlated with **`Name_petco`**.
 - Dropped column **`Name_slayter`** because it was too correlated with **`Name_hilda`**.
 - Dropped column **`Name_slocovski`** because it was too correlated with **`Name_selman`**.
 - Dropped column **`Name_slow`** because it was too correlated with **`Name_adelaide`**, **`Name_louch`**.
 - Dropped column **`Name_smart`** because it was too correlated with **`Name_montgomery`**.
 - Dropped column **`Name_smiljanic`** because it was too correlated with **`Name_mile`**.
 - Dropped column **`Name_sobey`** because it was too correlated with **`Name_hayden`**.
 - Dropped column **`Name_soto`** because it was too correlated with **`Name_josefa`**, **`Name_perez`**.
 - Dropped column **`Name_spedden`** because it was too correlated with **`Name_corning`**, **`Name_margaretta`**, **`Name_oakley`**.
 - Dropped column **`Name_stahelin`** because it was too correlated with **`Name_maeglin`**, **`Name_max`**.
 - Dropped column **`Name_stanio`** because it was too correlated with **`Name_gheorgheff`**.
 - Dropped column **`Name_stanlick`** because it was too correlated with **`Name_cordelia`**.
 - Dropped column **`Name_steffansson`** because it was too correlated with **`Name_bjornstrom`**, **`Name_hakan`**.
 - Dropped column **`Name_stefo`** because it was too correlated with **`Name_pavlovic`**.
 - Dropped column **`Name_stehli`** because it was too correlated with **`Name_maxmillian`**.
 - Dropped column **`Name_stoytcheff`** because it was too correlated with **`Name_ilia`**.
 - Dropped column **`Name_stoytcho`** because it was too correlated with **`Name_mionoff`**.
 - Dropped column **`Name_sutherland`** because it was too correlated with **`Name_christiana`**, **`Name_lady`**, **`Name_lucille`**.
 - Dropped column **`Name_suzette`** because it was too correlated with **`Name_parker`**.
 - Dropped column **`Name_sven`** because it was too correlated with **`Name_ivar`**.
 - Dropped column **`Name_swift`** because it was too correlated with **`Name_barron`**.
 - Dropped column **`Name_sylfven`** because it was too correlated with **`Name_lahtinen`**.
 - Dropped column **`Name_sylvia`** because it was too correlated with **`Name_harbaugh`**, **`Name_mae`**.
 - Dropped column **`Name_talmadge`** because it was too correlated with **`Name_astor`**, **`Name_force`**.
 - Dropped column **`Name_taylor`** because it was too correlated with **`Name_elmer`**.
 - Dropped column **`Name_thayer`** because it was too correlated with **`Name_borland`**.
 - Dropped column **`Name_the`** because it was too correlated with **`Name_countess`**, **`Name_dyer`**, **`Name_edwards`**, **`Name_noel`**, **`Name_of`**, **`Name_rothes`**.
 - Dropped column **`Name_thelander`** because it was too correlated with **`Name_eberhard`**.
 - Dropped column **`Name_theodore`** because it was too correlated with **`Name_mulder`**.
 - Dropped column **`Name_theodosia`** because it was too correlated with **`Name_kornelia`**.
 - Dropped column **`Name_thor`** because it was too correlated with **`Name_olsvigen`**.
 - Dropped column **`Name_thorilda`** because it was too correlated with **`Name_agda`**, **`Name_lindahl`**.
 - Dropped column **`Name_thuillard`** because it was too correlated with **`Name_jerwan`**, **`Name_marthe`**.
 - Dropped column **`Name_tido`** because it was too correlated with **`Name_rekic`**.
 - Dropped column **`Name_tillie`** because it was too correlated with **`Name_mandelbaum`**.
 - Dropped column **`Name_todor`** because it was too correlated with **`Name_sdycoff`**.
 - Dropped column **`Name_tomlin`** because it was too correlated with **`Name_portage`**.
 - Dropped column **`Name_torborg`** because it was too correlated with **`Name_danira`**.
 - Dropped column **`Name_toufik`** because it was too correlated with **`Name_nakli`**.
 - Dropped column **`Name_touma`** because it was too correlated with **`Name_darwis`**, **`Name_hanne`**.
 - Dropped column **`Name_towner`** because it was too correlated with **`Name_aline`**.
 - Dropped column **`Name_trembisky`** because it was too correlated with **`Name_berk`**, **`Name_pickard`**.
 - Dropped column **`Name_troupiansky`** because it was too correlated with **`Name_aaron`**, **`Name_moses`**.
 - Dropped column **`Name_troutt`** because it was too correlated with **`Name_celia`**, **`Name_edwina`**.
 - Dropped column **`Name_turcin`** because it was too correlated with **`Name_stjepan`**.
 - Dropped column **`Name_tyrell`** because it was too correlated with **`Name_cavendish`**.
 - Dropped column **`Name_ulrika`** because it was too correlated with **`Name_dahlberg`**, **`Name_gerda`**.
 - Dropped column **`Name_uruchurtu`** because it was too correlated with **`Name_don`**.
 - Dropped column **`Name_uscher`** because it was too correlated with **`Name_paulner`**.
 - Dropped column **`Name_vallejo`** because it was too correlated with **`Name_josefa`**, **`Name_perez`**, **`Name_soto`**.
 - Dropped column **`Name_vandemoortele`** because it was too correlated with **`Name_emelia`**, **`Name_julius`**.
 - Dropped column **`Name_vanden`** because it was too correlated with **`Name_steen`**.
 - Dropped column **`Name_vasil`** because it was too correlated with **`Name_plotcharsky`**.
 - Dropped column **`Name_velin`** because it was too correlated with **`Name_ohman`**.
 - Dropped column **`Name_vestrom`** because it was too correlated with **`Name_adolfina`**, **`Name_amanda`**, **`Name_hulda`**.
 - Dropped column **`Name_viktoria`** because it was too correlated with **`Name_agda`**, **`Name_lindahl`**, **`Name_thorilda`**.
 - Dropped column **`Name_vilhelmina`** because it was too correlated with **`Name_berg`**.
 - Dropped column **`Name_villiers`** because it was too correlated with **`Name_antonine`**, **`Name_berthe`**, **`Name_mayne`**.
 - Dropped column **`Name_viola`** because it was too correlated with **`Name_stina`**.
 - Dropped column **`Name_virginia`** because it was too correlated with **`Name_emanuel`**.
 - Dropped column **`Name_vivian`** because it was too correlated with **`Name_drew`**, **`Name_lulu`**.
 - Dropped column **`Name_vovk`** because it was too correlated with **`Name_janko`**.
 - Dropped column **`Name_waddington`** because it was too correlated with **`Name_sedgwick`**.
 - Dropped column **`Name_waelens`** because it was too correlated with **`Name_achille`**.
 - Dropped column **`Name_waldo`** because it was too correlated with **`Name_daniels`**.
 - Dropped column **`Name_wallach`** because it was too correlated with **`Name_irene`**.
 - Dropped column **`Name_walle`** because it was too correlated with **`Name_cyriel`**, **`Name_nestor`**.
 - Dropped column **`Name_warner`** because it was too correlated with **`Name_marvin`**.
 - Dropped column **`Name_warren`** because it was too correlated with **`Name_atkinson`**, **`Name_manley`**.
 - Dropped column **`Name_watt`** because it was too correlated with **`Name_inglis`**, **`Name_milne`**.
 - Dropped column **`Name_weart`** because it was too correlated with **`Name_blackwell`**.
 - Dropped column **`Name_weisz`** because it was too correlated with **`Name_francoise`**, **`Name_leopold`**, **`Name_pede`**.
 - Dropped column **`Name_welles`** because it was too correlated with **`Name_barron`**, **`Name_swift`**.
 - Dropped column **`Name_wells`** because it was too correlated with **`Name_joan`**.
 - Dropped column **`Name_wendla`** because it was too correlated with **`Name_heininen`**.
 - Dropped column **`Name_werner`** because it was too correlated with **`Name_salonen`**.
 - Dropped column **`Name_widener`** because it was too correlated with **`Name_elkins`**.
 - Dropped column **`Name_wilhelmina`** because it was too correlated with **`Name_helena`**.
 - Dropped column **`Name_wilkinson`** because it was too correlated with **`Name_faunthorpe`**, **`Name_lizzie`**.
 - Dropped column **`Name_willingham`** because it was too correlated with **`Name_archibald`**, **`Name_butt`**.
 - Dropped column **`Name_wills`** because it was too correlated with **`Name_leitch`**.
 - Dropped column **`Name_windelov`** because it was too correlated with **`Name_einar`**.
 - Dropped column **`Name_winnie`** because it was too correlated with **`Name_celia`**, **`Name_edwina`**, **`Name_troutt`**.
 - Dropped column **`Name_wiseman`** because it was too correlated with **`Name_phillippe`**.
 - Dropped column **`Name_wood`** because it was too correlated with **`Name_archie`**, **`Name_frost`**.
 - Dropped column **`Name_wyckoff`** because it was too correlated with **`Name_der`**, **`Name_hoef`**.
 - Dropped column **`Name_yarred`** because it was too correlated with **`Name_nicola`**.
 - Dropped column **`Name_yasbeck`** because it was too correlated with **`Name_antoni`**.
 - Dropped column **`Name_yoto`** because it was too correlated with **`Name_danoff`**.
 - Dropped column **`Name_young`** because it was too correlated with **`Name_grice`**.
 - Dropped column **`Name_yousif`** because it was too correlated with **`Name_wazli`**.
 - Dropped column **`Name_yrois`** because it was too correlated with **`Name_harbeck`**, **`Name_henriette`**.
 - Dropped column **`Name_zebley`** because it was too correlated with **`Name_elmer`**, **`Name_taylor`**.
 - Dropped column **`Name_zenni`** because it was too correlated with **`Name_fahim`**, **`Name_leeni`**.
 - Dropped column **`Ticket_10482`** because it was too correlated with **`Name_dorking`**.
 - Dropped column **`Ticket_110413`** because it was too correlated with **`Name_taussig`**.
 - Dropped column **`Ticket_110564`** because it was too correlated with **`Name_bjornstrom`**, **`Name_hakan`**, **`Name_steffansson`**.
 - Dropped column **`Ticket_110813`** because it was too correlated with **`Name_atkinson`**, **`Name_manley`**, **`Name_warren`**.
 - Dropped column **`Ticket_111240`** because it was too correlated with **`Name_der`**, **`Name_hoef`**, **`Name_wyckoff`**.
 - Dropped column **`Ticket_111320`** because it was too correlated with **`Name_gee`**.
 - Dropped column **`Ticket_111361`** because it was too correlated with **`Name_hippach`**.
 - Dropped column **`Ticket_111369`** because it was too correlated with **`Name_behr`**, **`Name_howell`**.
 - Dropped column **`Ticket_111426`** because it was too correlated with **`Name_haven`**, **`Name_homer`**.
 - Dropped column **`Ticket_111427`** because it was too correlated with **`Name_brayton`**.
 - Dropped column **`Ticket_111428`** because it was too correlated with **`Name_hallace`**, **`Name_rolmane`**, **`Name_romaine`**.
 - Dropped column **`Ticket_112052`** because it was too correlated with **`Name_marsh`**, **`Name_parr`**.
 - Dropped column **`Ticket_112058`** because it was too correlated with **`Name_fry`**.
 - Dropped column **`Ticket_112059`** because it was too correlated with **`Name_harrison`**.
 - Dropped column **`Ticket_11206`** because it was too correlated with **`Name_alfonzo`**, **`Name_meo`**.
 - Dropped column **`Ticket_112277`** because it was too correlated with **`Name_blank`**.
 - Dropped column **`Ticket_112379`** because it was too correlated with **`Name_brewe`**, **`Name_jackson`**.
 - Dropped column **`Ticket_113028`** because it was too correlated with **`Name_klaber`**.
 - Dropped column **`Ticket_113043`** because it was too correlated with **`Name_austen`**, **`Name_partner`**.
 - Dropped column **`Ticket_113050`** because it was too correlated with **`Name_archibald`**, **`Name_butt`**, **`Name_willingham`**.
 - Dropped column **`Ticket_113051`** because it was too correlated with **`Name_foreman`**, **`Name_laventall`**.
 - Dropped column **`Ticket_113059`** because it was too correlated with **`Name_carrau`**, **`Name_francisco`**.
 - Dropped column **`Ticket_113501`** because it was too correlated with **`Name_clyde`**, **`Name_long`**, **`Name_milton`**.
 - Dropped column **`Ticket_113503`** because it was too correlated with **`Name_elkins`**, **`Name_widener`**.
 - Dropped column **`Ticket_113505`** because it was too correlated with **`Name_bowerman`**.
 - Dropped column **`Ticket_113509`** because it was too correlated with **`Name_cornelius`**, **`Name_engelhart`**, **`Name_ostby`**.
 - Dropped column **`Ticket_113510`** because it was too correlated with **`Name_fellows`**, **`Name_fletcher`**, **`Name_lambert`**.
 - Dropped column **`Ticket_113514`** because it was too correlated with **`Name_stead`**.
 - Dropped column **`Ticket_113572`** because it was too correlated with **`Embarked_nan`**.
 - Dropped column **`Ticket_113767`** because it was too correlated with **`Name_rood`**, **`Name_roscoe`**.
 - Dropped column **`Ticket_113773`** because it was too correlated with **`Name_marvin`**, **`Name_warner`**.
 - Dropped column **`Ticket_113776`** because it was too correlated with **`Name_pears`**.
 - Dropped column **`Ticket_113783`** because it was too correlated with **`Name_bonnell`**.
 - Dropped column **`Ticket_113784`** because it was too correlated with **`Name_blackwell`**, **`Name_weart`**.
 - Dropped column **`Ticket_113786`** because it was too correlated with **`Name_peuchen`**.
 - Dropped column **`Ticket_113787`** because it was too correlated with **`Name_markland`**, **`Name_molson`**.
 - Dropped column **`Ticket_113788`** because it was too correlated with **`Name_sloper`**.
 - Dropped column **`Ticket_113789`** because it was too correlated with **`Name_holverson`**.
 - Dropped column **`Ticket_113792`** because it was too correlated with **`Name_montgomery`**, **`Name_smart`**.
 - Dropped column **`Ticket_113794`** because it was too correlated with **`Name_kimber`**, **`Name_seward`**.
 - Dropped column **`Ticket_113796`** because it was too correlated with **`Name_harrington`**.
 - Dropped column **`Ticket_113800`** because it was too correlated with **`Name_weir`**.
 - Dropped column **`Ticket_113803`** because it was too correlated with **`Name_futrelle`**, **`Name_heath`**.
 - Dropped column **`Ticket_113806`** because it was too correlated with **`Name_chambers`**.
 - Dropped column **`Ticket_1166`** because it was too correlated with **`Name_bateman`**.
 - Dropped column **`Ticket_11668`** because it was too correlated with **`Name_turpin`**.
 - Dropped column **`Ticket_11751`** because it was too correlated with **`Name_beckwith`**.
 - Dropped column **`Ticket_11752`** because it was too correlated with **`Name_newsom`**.
 - Dropped column **`Ticket_11753`** because it was too correlated with **`Name_edwin`**, **`Name_kimball`**.
 - Dropped column **`Ticket_11755`** because it was too correlated with **`Name_christiana`**, **`Name_lady`**, **`Name_lucille`**, **`Name_sutherland`**.
 - Dropped column **`Ticket_11765`** because it was too correlated with **`Name_achilles`**, **`Name_harder`**.
 - Dropped column **`Ticket_11769`** because it was too correlated with **`Name_appleton`**, **`Name_dale`**, **`Name_lamson`**.
 - Dropped column **`Ticket_11771`** because it was too correlated with **`Name_kent`**.
 - Dropped column **`Ticket_11774`** because it was too correlated with **`Name_marechal`**, **`Name_pierre`**.
 - Dropped column **`Ticket_11813`** because it was too correlated with **`Name_albina`**, **`Name_bazzani`**.
 - Dropped column **`Ticket_11967`** because it was too correlated with **`Name_bishop`**, **`Name_dickinson`**.
 - Dropped column **`Ticket_12460`** because it was too correlated with **`Name_andy`**.
 - Dropped column **`Ticket_12750`** because it was too correlated with **`Name_davidson`**.
 - Dropped column **`Ticket_13032`** because it was too correlated with **`Name_charters`**.
 - Dropped column **`Ticket_13049`** because it was too correlated with **`Name_ross`**.
 - Dropped column **`Ticket_13213`** because it was too correlated with **`Name_blumer`**, **`Name_oberst`**, **`Name_simonius`**.
 - Dropped column **`Ticket_13214`** because it was too correlated with **`Name_maeglin`**, **`Name_max`**, **`Name_stahelin`**.
 - Dropped column **`Ticket_13507`** because it was too correlated with **`Name_baird`**, **`Name_silvey`**.
 - Dropped column **`Ticket_13509`** because it was too correlated with **`Name_millet`**.
 - Dropped column **`Ticket_13528`** because it was too correlated with **`Name_cameron`**, **`Name_clear`**.
 - Dropped column **`Ticket_13531`** because it was too correlated with **`Name_toomey`**.
 - Dropped column **`Ticket_13567`** because it was too correlated with **`Name_maxmillian`**, **`Name_stehli`**.
 - Dropped column **`Ticket_13568`** because it was too correlated with **`Name_margaritha`**.
 - Dropped column **`Ticket_14258`** because it was too correlated with **`Name_lucy`**, **`Name_ridsdale`**.
 - Dropped column **`Ticket_14263`** because it was too correlated with **`Name_coleridge`**.
 - Dropped column **`Ticket_14311`** because it was too correlated with **`Name_driscoll`**.
 - Dropped column **`Ticket_14313`** because it was too correlated with **`Name_jermyn`**.
 - Dropped column **`Ticket_14885`** because it was too correlated with **`Name_ilett`**.
 - Dropped column **`Ticket_14973`** because it was too correlated with **`Name_eliezer`**, **`Name_gilinski`**.
 - Dropped column **`Ticket_16988`** because it was too correlated with **`Name_hawksford`**.
 - Dropped column **`Ticket_17248`** because it was too correlated with **`Name_reeves`**.
 - Dropped column **`Ticket_17318`** because it was too correlated with **`Name_baumann`**.
 - Dropped column **`Ticket_17369`** because it was too correlated with **`Name_fridtjof`**, **`Name_madsen`**.
 - Dropped column **`Ticket_17453`** because it was too correlated with **`Name_goldenberg`**.
 - Dropped column **`Ticket_17463`** because it was too correlated with **`Name_mccarthy`**.
 - Dropped column **`Ticket_17464`** because it was too correlated with **`Name_kenyon`**.
 - Dropped column **`Ticket_17465`** because it was too correlated with **`Name_farnham`**, **`Name_leader`**.
 - Dropped column **`Ticket_17466`** because it was too correlated with **`Name_barron`**, **`Name_swift`**, **`Name_welles`**.
 - Dropped column **`Ticket_17473`** because it was too correlated with **`Name_mcgough`**.
 - Dropped column **`Ticket_17475`** because it was too correlated with **`Name_silverthorne`**.
 - Dropped column **`Ticket_17476`** because it was too correlated with **`Name_calderhead`**, **`Name_pennington`**.
 - Dropped column **`Ticket_1748`** because it was too correlated with **`Name_lehmann`**.
 - Dropped column **`Ticket_17482`** because it was too correlated with **`Name_antonine`**, **`Name_berthe`**, **`Name_mayne`**, **`Name_villiers`**.
 - Dropped column **`Ticket_17483`** because it was too correlated with **`Name_farthing`**.
 - Dropped column **`Ticket_17558`** because it was too correlated with **`Name_baxter`**.
 - Dropped column **`Ticket_17585`** because it was too correlated with **`Name_maybelle`**.
 - Dropped column **`Ticket_17590`** because it was too correlated with **`Name_roebling`**.
 - Dropped column **`Ticket_17592`** because it was too correlated with **`Name_conover`**, **`Name_lines`**.
 - Dropped column **`Ticket_17595`** because it was too correlated with **`Name_isham`**.
 - Dropped column **`Ticket_17596`** because it was too correlated with **`Name_natsch`**.
 - Dropped column **`Ticket_17599`** because it was too correlated with **`Name_briggs`**, **`Name_cumings`**.
 - Dropped column **`Ticket_17600`** because it was too correlated with **`Name_fisher`**.
 - Dropped column **`Ticket_17601`** because it was too correlated with **`Name_don`**, **`Name_uruchurtu`**.
 - Dropped column **`Ticket_17603`** because it was too correlated with **`Name_barrett`**, **`Name_rothschild`**.
 - Dropped column **`Ticket_17604`** because it was too correlated with **`Name_edgar`**.
 - Dropped column **`Ticket_17605`** because it was too correlated with **`Name_stewart`**.
 - Dropped column **`Ticket_17608`** because it was too correlated with **`Name_ryerson`**.
 - Dropped column **`Ticket_17609`** because it was too correlated with **`Name_artagaveytia`**, **`Name_ramon`**.
 - Dropped column **`Ticket_17611`** because it was too correlated with **`Name_frauenthal`**.
 - Dropped column **`Ticket_17612`** because it was too correlated with **`Name_ervin`**, **`Name_lewy`**.
 - Dropped column **`Ticket_17754`** because it was too correlated with **`Name_goldschmidt`**.
 - Dropped column **`Ticket_17756`** because it was too correlated with **`Name_compton`**, **`Name_rebecca`**, **`Name_sara`**.
 - Dropped column **`Ticket_17758`** because it was too correlated with **`Name_castellana`**, **`Name_penasco`**, **`Name_satode`**.
 - Dropped column **`Ticket_17759`** because it was too correlated with **`Name_greenfield`**.
 - Dropped column **`Ticket_17764`** because it was too correlated with **`Name_clinch`**.
 - Dropped column **`Ticket_18509`** because it was too correlated with **`Name_somerton`**.
 - Dropped column **`Ticket_18723`** because it was too correlated with **`Name_johannesson`**, **`Name_kvillner`**.
 - Dropped column **`Ticket_19928`** because it was too correlated with **`Name_minahan`**.
 - Dropped column **`Ticket_19943`** because it was too correlated with **`Name_maxfield`**.
 - Dropped column **`Ticket_19947`** because it was too correlated with **`Name_woolner`**.
 - Dropped column **`Ticket_19950`** because it was too correlated with **`Name_fortune`**.
 - Dropped column **`Ticket_19972`** because it was too correlated with **`Name_jonkheer`**, **`Name_reuchlin`**.
 - Dropped column **`Ticket_19988`** because it was too correlated with **`Name_adolphe`**, **`Name_saalfeld`**.
 - Dropped column **`Ticket_19996`** because it was too correlated with **`Name_elmer`**, **`Name_taylor`**, **`Name_zebley`**.
 - Dropped column **`Ticket_2003`** because it was too correlated with **`Name_winfield`**.
 - Dropped column **`Ticket_20589`** because it was too correlated with **`Name_rush`**.
 - Dropped column **`Ticket_2079`** because it was too correlated with **`Name_mallet`**.
 - Dropped column **`Ticket_211536`** because it was too correlated with **`Name_juozas`**, **`Name_montvila`**.
 - Dropped column **`Ticket_21172`** because it was too correlated with **`Name_dennis`**.
 - Dropped column **`Ticket_21173`** because it was too correlated with **`Name_lovell`**.
 - Dropped column **`Ticket_21174`** because it was too correlated with **`Name_perkin`**.
 - Dropped column **`Ticket_2123`** because it was too correlated with **`Name_laroche`**.
 - Dropped column **`Ticket_2131`** because it was too correlated with **`Name_pernot`**.
 - Dropped column **`Ticket_2133`** because it was too correlated with **`Name_emile`**.
 - Dropped column **`Ticket_2144`** because it was too correlated with **`Name_goodwin`**.
 - Dropped column **`Ticket_21440`** because it was too correlated with **`Name_green`**.
 - Dropped column **`Ticket_2146`** because it was too correlated with **`Name_manent`**, **`Name_padro`**.
 - Dropped column **`Ticket_2149`** because it was too correlated with **`Name_asuncion`**, **`Name_duran`**, **`Name_more`**.
 - Dropped column **`Ticket_2151`** because it was too correlated with **`Name_saundercock`**.
 - Dropped column **`Ticket_2152`** because it was too correlated with **`Name_cann`**.
 - Dropped column **`Ticket_2163`** because it was too correlated with **`Name_levy`**.
 - Dropped column **`Ticket_2167`** because it was too correlated with **`Name_carlo`**, **`Name_del`**, **`Name_sebastiano`**.
 - Dropped column **`Ticket_219533`** because it was too correlated with **`Name_kirkland`**.
 - Dropped column **`Ticket_220367`** because it was too correlated with **`Name_bracken`**.
 - Dropped column **`Ticket_2223`** because it was too correlated with **`Name_allum`**.
 - Dropped column **`Ticket_223596`** because it was too correlated with **`Name_fannie`**.
 - Dropped column **`Ticket_226875`** because it was too correlated with **`Name_angle`**.
 - Dropped column **`Ticket_228414`** because it was too correlated with **`Name_francoise`**, **`Name_leopold`**, **`Name_pede`**, **`Name_weisz`**.
 - Dropped column **`Ticket_229236`** because it was too correlated with **`Name_fox`**, **`Name_hubert`**.
 - Dropped column **`Ticket_230080`** because it was too correlated with **`Name_navratil`**.
 - Dropped column **`Ticket_230136`** because it was too correlated with **`Name_becker`**.
 - Dropped column **`Ticket_230433`** because it was too correlated with **`Name_parrish`**.
 - Dropped column **`Ticket_230434`** because it was too correlated with **`Name_encarnacion`**, **`Name_ms`**, **`Name_reynaldo`**.
 - Dropped column **`Ticket_2314`** because it was too correlated with **`Name_zillah`**.
 - Dropped column **`Ticket_2315`** because it was too correlated with **`Name_dean`**.
 - Dropped column **`Ticket_231919`** because it was too correlated with **`Name_doling`**.
 - Dropped column **`Ticket_231945`** because it was too correlated with **`Name_edgardo`**.
 - Dropped column **`Ticket_233639`** because it was too correlated with **`Name_aaron`**, **`Name_moses`**, **`Name_troupiansky`**.
 - Dropped column **`Ticket_233866`** because it was too correlated with **`Name_gill`**.
 - Dropped column **`Ticket_2343`** because it was too correlated with **`Name_sage`**.
 - Dropped column **`Ticket_234360`** because it was too correlated with **`Name_milling`**.
 - Dropped column **`Ticket_234604`** because it was too correlated with **`Name_pinsky`**.
 - Dropped column **`Ticket_234686`** because it was too correlated with **`Name_butler`**, **`Name_fenton`**.
 - Dropped column **`Ticket_234818`** because it was too correlated with **`Name_hilda`**, **`Name_slayter`**.
 - Dropped column **`Ticket_23567`** because it was too correlated with **`Name_rogers`**.
 - Dropped column **`Ticket_236171`** because it was too correlated with **`Name_fahlstrom`**, **`Name_jonas`**.
 - Dropped column **`Ticket_236852`** because it was too correlated with **`Name_bystrom`**, **`Name_karolina`**.
 - Dropped column **`Ticket_236853`** because it was too correlated with **`Name_bryhl`**, **`Name_gottfrid`**, **`Name_kurt`**.
 - Dropped column **`Ticket_237442`** because it was too correlated with **`Name_sjostedt`**.
 - Dropped column **`Ticket_237565`** because it was too correlated with **`Name_denzil`**, **`Name_jarvis`**.
 - Dropped column **`Ticket_237671`** because it was too correlated with **`Name_clemmer`**, **`Name_funk`**.
 - Dropped column **`Ticket_237736`** because it was too correlated with **`Name_nasser`**, **`Name_nicholas`**.
 - Dropped column **`Ticket_237789`** because it was too correlated with **`Name_julie`**, **`Name_rachel`**.
 - Dropped column **`Ticket_237798`** because it was too correlated with **`Name_hosono`**, **`Name_masabumi`**.
 - Dropped column **`Ticket_239854`** because it was too correlated with **`Name_archie`**, **`Name_frost`**, **`Name_wood`**.
 - Dropped column **`Ticket_239855`** because it was too correlated with **`Name_knight`**.
 - Dropped column **`Ticket_239856`** because it was too correlated with **`Name_ennis`**, **`Name_hastings`**.
 - Dropped column **`Ticket_240929`** because it was too correlated with **`Name_trout`**.
 - Dropped column **`Ticket_243847`** because it was too correlated with **`Name_jacobsohn`**.
 - Dropped column **`Ticket_243880`** because it was too correlated with **`Name_garside`**.
 - Dropped column **`Ticket_244252`** because it was too correlated with **`Name_courtenay`**.
 - Dropped column **`Ticket_244270`** because it was too correlated with **`Name_wilhelms`**.
 - Dropped column **`Ticket_244278`** because it was too correlated with **`Name_pain`**.
 - Dropped column **`Ticket_244310`** because it was too correlated with **`Name_byles`**, **`Name_davids`**, **`Name_roussel`**.
 - Dropped column **`Ticket_244358`** because it was too correlated with **`Name_sharp`**.
 - Dropped column **`Ticket_244361`** because it was too correlated with **`Name_sedgwick`**, **`Name_waddington`**.
 - Dropped column **`Ticket_244367`** because it was too correlated with **`Name_kantor`**, **`Name_sinai`**.
 - Dropped column **`Ticket_24579`** because it was too correlated with **`Name_wheadon`**.
 - Dropped column **`Ticket_24580`** because it was too correlated with **`Name_mitchell`**.
 - Dropped column **`Ticket_2466`** because it was too correlated with **`Name_howard`**.
 - Dropped column **`Ticket_248698`** because it was too correlated with **`Name_beesley`**.
 - Dropped column **`Ticket_248706`** because it was too correlated with **`Name_hewlett`**, **`Name_kingcome`**.
 - Dropped column **`Ticket_248733`** because it was too correlated with **`Name_mildred`**.
 - Dropped column **`Ticket_248738`** because it was too correlated with **`Name_caldwell`**.
 - Dropped column **`Ticket_248740`** because it was too correlated with **`Name_collander`**.
 - Dropped column **`Ticket_248747`** because it was too correlated with **`Name_harbeck`**, **`Name_henriette`**, **`Name_yrois`**.
 - Dropped column **`Ticket_250643`** because it was too correlated with **`Name_hodges`**, **`Name_price`**.
 - Dropped column **`Ticket_250644`** because it was too correlated with **`Name_mellinger`**.
 - Dropped column **`Ticket_250646`** because it was too correlated with **`Name_givard`**, **`Name_kristensen`**.
 - Dropped column **`Ticket_250648`** because it was too correlated with **`Name_sinkkonen`**.
 - Dropped column **`Ticket_250649`** because it was too correlated with **`Name_hamalainen`**.
 - Dropped column **`Ticket_250651`** because it was too correlated with **`Name_lahtinen`**, **`Name_sylfven`**.
 - Dropped column **`Ticket_250652`** because it was too correlated with **`Name_karoliina`**, **`Name_lyyli`**, **`Name_silven`**.
 - Dropped column **`Ticket_250653`** because it was too correlated with **`Name_hale`**.
 - Dropped column **`Ticket_250655`** because it was too correlated with **`Name_marshall`**.
 - Dropped column **`Ticket_2620`** because it was too correlated with **`Name_fahim`**, **`Name_leeni`**, **`Name_zenni`**.
 - Dropped column **`Ticket_2623`** because it was too correlated with **`Name_badt`**, **`Name_mohamed`**.
 - Dropped column **`Ticket_2624`** because it was too correlated with **`Name_lahoud`**, **`Name_sarkis`**.
 - Dropped column **`Ticket_2625`** because it was too correlated with **`Name_assad`**.
 - Dropped column **`Ticket_2626`** because it was too correlated with **`Name_mantoura`**, **`Name_moussa`**.
 - Dropped column **`Ticket_2628`** because it was too correlated with **`Name_youseff`**.
 - Dropped column **`Ticket_2629`** because it was too correlated with **`Name_raihed`**.
 - Dropped column **`Ticket_2631`** because it was too correlated with **`Name_chehab`**, **`Name_emir`**, **`Name_farred`**.
 - Dropped column **`Ticket_26360`** because it was too correlated with **`Name_quick`**.
 - Dropped column **`Ticket_2641`** because it was too correlated with **`Name_nakli`**, **`Name_toufik`**.
 - Dropped column **`Ticket_2647`** because it was too correlated with **`Name_wazli`**, **`Name_yousif`**.
 - Dropped column **`Ticket_2648`** because it was too correlated with **`Name_betros`**.
 - Dropped column **`Ticket_2649`** because it was too correlated with **`Name_fatima`**, **`Name_masselmani`**.
 - Dropped column **`Ticket_2650`** because it was too correlated with **`Name_darwis`**, **`Name_hanne`**, **`Name_touma`**.
 - Dropped column **`Ticket_2651`** because it was too correlated with **`Name_nicola`**, **`Name_yarred`**.
 - Dropped column **`Ticket_2653`** because it was too correlated with **`Name_nakid`**.
 - Dropped column **`Ticket_265302`** because it was too correlated with **`Name_ole`**.
 - Dropped column **`Ticket_2659`** because it was too correlated with **`Name_antoni`**, **`Name_yasbeck`**.
 - Dropped column **`Ticket_2661`** because it was too correlated with **`Name_moubarek`**.
 - Dropped column **`Ticket_2662`** because it was too correlated with **`Name_samaan`**.
 - Dropped column **`Ticket_2663`** because it was too correlated with **`Name_assi`**, **`Name_barah`**.
 - Dropped column **`Ticket_2665`** because it was too correlated with **`Name_zabour`**.
 - Dropped column **`Ticket_2666`** because it was too correlated with **`Name_baclini`**.
 - Dropped column **`Ticket_2667`** because it was too correlated with **`Name_kiamie`**, **`Name_najib`**.
 - Dropped column **`Ticket_2669`** because it was too correlated with **`Name_orsen`**, **`Name_sirayanian`**.
 - Dropped column **`Ticket_26707`** because it was too correlated with **`Name_hold`**.
 - Dropped column **`Ticket_2672`** because it was too correlated with **`Name_khalil`**.
 - Dropped column **`Ticket_2673`** because it was too correlated with **`Name_abbott`**.
 - Dropped column **`Ticket_2674`** because it was too correlated with **`Name_dibo`**.
 - Dropped column **`Ticket_2677`** because it was too correlated with **`Name_mamee`**.
 - Dropped column **`Ticket_2680`** because it was too correlated with **`Name_apostolos`**, **`Name_chronopoulos`**.
 - Dropped column **`Ticket_2683`** because it was too correlated with **`Name_lemberopolous`**.
 - Dropped column **`Ticket_2685`** because it was too correlated with **`Name_ibrahim`**, **`Name_shawah`**.
 - Dropped column **`Ticket_2686`** because it was too correlated with **`Name_doharr`**.
 - Dropped column **`Ticket_2687`** because it was too correlated with **`Name_ayoub`**, **`Name_banoura`**.
 - Dropped column **`Ticket_2689`** because it was too correlated with **`Name_caram`**.
 - Dropped column **`Ticket_2693`** because it was too correlated with **`Name_mansour`**.
 - Dropped column **`Ticket_2694`** because it was too correlated with **`Name_sleiman`**.
 - Dropped column **`Ticket_2697`** because it was too correlated with **`Name_mansouer`**, **`Name_novel`**.
 - Dropped column **`Ticket_2700`** because it was too correlated with **`Name_fared`**, **`Name_kassem`**.
 - Dropped column **`Ticket_27042`** because it was too correlated with **`Name_algernon`**, **`Name_barkworth`**.
 - Dropped column **`Ticket_27849`** because it was too correlated with **`Name_buss`**.
 - Dropped column **`Ticket_28134`** because it was too correlated with **`Name_giles`**.
 - Dropped column **`Ticket_2816`** because it was too correlated with **`Name_maisner`**, **`Name_simon`**.
 - Dropped column **`Ticket_2817`** because it was too correlated with **`Name_peduzzi`**.
 - Dropped column **`Ticket_28206`** because it was too correlated with **`Name_slemen`**.
 - Dropped column **`Ticket_28213`** because it was too correlated with **`Name_otter`**.
 - Dropped column **`Ticket_28220`** because it was too correlated with **`Name_drew`**, **`Name_lulu`**, **`Name_vivian`**.
 - Dropped column **`Ticket_28228`** because it was too correlated with **`Name_matthews`**.
 - Dropped column **`Ticket_28424`** because it was too correlated with **`Name_carbines`**.
 - Dropped column **`Ticket_28425`** because it was too correlated with **`Name_berriman`**.
 - Dropped column **`Ticket_28551`** because it was too correlated with **`Name_ball`**.
 - Dropped column **`Ticket_28664`** because it was too correlated with **`Name_gale`**, **`Name_shadrach`**.
 - Dropped column **`Ticket_28665`** because it was too correlated with **`Name_pengelly`**.
 - Dropped column **`Ticket_29011`** because it was too correlated with **`Name_moraweck`**.
 - Dropped column **`Ticket_2908`** because it was too correlated with **`Name_beane`**.
 - Dropped column **`Ticket_29103`** because it was too correlated with **`Name_joan`**, **`Name_wells`**.
 - Dropped column **`Ticket_29105`** because it was too correlated with **`Name_eliza`**, **`Name_needs`**.
 - Dropped column **`Ticket_29108`** because it was too correlated with **`Name_bailey`**, **`Name_percy`**.
 - Dropped column **`Ticket_29178`** because it was too correlated with **`Name_hayden`**, **`Name_sobey`**.
 - Dropped column **`Ticket_2926`** because it was too correlated with **`Name_faunthorpe`**, **`Name_lizzie`**, **`Name_wilkinson`**.
 - Dropped column **`Ticket_29395`** because it was too correlated with **`Name_nye`**, **`Name_ramell`**.
 - Dropped column **`Ticket_29566`** because it was too correlated with **`Name_leyson`**.
 - Dropped column **`Ticket_29751`** because it was too correlated with **`Name_eitemiller`**, **`Name_floyd`**.
 - Dropped column **`Ticket_3085`** because it was too correlated with **`Name_adelaide`**, **`Name_louch`**, **`Name_slow`**.
 - Dropped column **`Ticket_3101264`** because it was too correlated with **`Name_johanson`**.
 - Dropped column **`Ticket_3101265`** because it was too correlated with **`Name_sjoblom`**.
 - Dropped column **`Ticket_3101267`** because it was too correlated with **`Name_wiklund`**.
 - Dropped column **`Ticket_3101269`** because it was too correlated with **`Name_sundman`**.
 - Dropped column **`Ticket_3101271`** because it was too correlated with **`Name_ilmakangas`**, **`Name_pieta`**.
 - Dropped column **`Ticket_3101272`** because it was too correlated with **`Name_aijo`**, **`Name_antino`**, **`Name_iisakki`**, **`Name_nirva`**.
 - Dropped column **`Ticket_3101273`** because it was too correlated with **`Name_rintamaki`**.
 - Dropped column **`Ticket_3101274`** because it was too correlated with **`Name_erland`**, **`Name_kallio`**, **`Name_nikolai`**.
 - Dropped column **`Ticket_3101275`** because it was too correlated with **`Name_alexanteri`**, **`Name_maenpaa`**.
 - Dropped column **`Ticket_3101276`** because it was too correlated with **`Name_vilhelm`**.
 - Dropped column **`Ticket_3101277`** because it was too correlated with **`Name_birger`**.
 - Dropped column **`Ticket_3101278`** because it was too correlated with **`Name_backstrom`**.
 - Dropped column **`Ticket_3101279`** because it was too correlated with **`Name_hakkarainen`**, **`Name_pekka`**, **`Name_pietari`**.
 - Dropped column **`Ticket_3101280`** because it was too correlated with **`Name_sivola`**.
 - Dropped column **`Ticket_3101281`** because it was too correlated with **`Name_erna`**.
 - Dropped column **`Ticket_3101282`** because it was too correlated with **`Name_heikkinen`**, **`Name_laina`**.
 - Dropped column **`Ticket_3101283`** because it was too correlated with **`Name_eliina`**, **`Name_honkanen`**.
 - Dropped column **`Ticket_3101285`** because it was too correlated with **`Name_lindqvist`**.
 - Dropped column **`Ticket_3101286`** because it was too correlated with **`Name_eiriik`**.
 - Dropped column **`Ticket_3101287`** because it was too correlated with **`Name_alhomaki`**, **`Name_ilmari`**, **`Name_rudolf`**.
 - Dropped column **`Ticket_3101288`** because it was too correlated with **`Name_stranden`**.
 - Dropped column **`Ticket_3101289`** because it was too correlated with **`Name_niskanen`**.
 - Dropped column **`Ticket_3101290`** because it was too correlated with **`Name_heininen`**, **`Name_wendla`**.
 - Dropped column **`Ticket_3101292`** because it was too correlated with **`Name_leinonen`**.
 - Dropped column **`Ticket_3101293`** because it was too correlated with **`Name_tikkanen`**.
 - Dropped column **`Ticket_3101294`** because it was too correlated with **`Name_pekoniemi`**.
 - Dropped column **`Ticket_3101295`** because it was too correlated with **`Name_panula`**.
 - Dropped column **`Ticket_3101296`** because it was too correlated with **`Name_salonen`**, **`Name_werner`**.
 - Dropped column **`Ticket_3101298`** because it was too correlated with **`Name_hildur`**, **`Name_hirvonen`**.
 - Dropped column **`Ticket_3101305`** because it was too correlated with **`Name_jardin`**, **`Name_jose`**, **`Name_neto`**.
 - Dropped column **`Ticket_3101306`** because it was too correlated with **`Name_estanslas`**, **`Name_goncalves`**.
 - Dropped column **`Ticket_3101307`** because it was too correlated with **`Name_coelho`**, **`Name_domingos`**, **`Name_fernandeo`**.
 - Dropped column **`Ticket_3101310`** because it was too correlated with **`Name_adola`**, **`Name_asim`**.
 - Dropped column **`Ticket_3101311`** because it was too correlated with **`Name_ahmed`**.
 - Dropped column **`Ticket_3101317`** because it was too correlated with **`Name_einar`**, **`Name_windelov`**.
 - Dropped column **`Ticket_31026`** because it was too correlated with **`Name_rugg`**.
 - Dropped column **`Ticket_31027`** because it was too correlated with **`Name_renouf`**.
 - Dropped column **`Ticket_31028`** because it was too correlated with **`Name_gavey`**.
 - Dropped column **`Ticket_312991`** because it was too correlated with **`Name_moss`**.
 - Dropped column **`Ticket_312992`** because it was too correlated with **`Name_birkeland`**, **`Name_monsen`**.
 - Dropped column **`Ticket_312993`** because it was too correlated with **`Name_knud`**, **`Name_paust`**, **`Name_rommetvedt`**.
 - Dropped column **`Ticket_315037`** because it was too correlated with **`Name_mile`**, **`Name_smiljanic`**.
 - Dropped column **`Ticket_315082`** because it was too correlated with **`Name_zimmerman`**.
 - Dropped column **`Ticket_315086`** because it was too correlated with **`Name_petar`**.
 - Dropped column **`Ticket_315088`** because it was too correlated with **`Name_dimic`**, **`Name_jovan`**.
 - Dropped column **`Ticket_315090`** because it was too correlated with **`Name_culumovic`**, **`Name_jeso`**.
 - Dropped column **`Ticket_315093`** because it was too correlated with **`Name_jovo`**.
 - Dropped column **`Ticket_315097`** because it was too correlated with **`Name_pasic`**.
 - Dropped column **`Ticket_315098`** because it was too correlated with **`Name_lulic`**, **`Name_nikola`**.
 - Dropped column **`Ticket_315151`** because it was too correlated with **`Name_vincenz`**.
 - Dropped column **`Ticket_315153`** because it was too correlated with **`Name_heilmann`**, **`Name_luise`**.
 - Dropped column **`Ticket_31921`** because it was too correlated with **`Name_collyer`**.
 - Dropped column **`Ticket_3235`** because it was too correlated with **`Name_murdlin`**.
 - Dropped column **`Ticket_323592`** because it was too correlated with **`Name_keefe`**.
 - Dropped column **`Ticket_323951`** because it was too correlated with **`Name_beavan`**.
 - Dropped column **`Ticket_324669`** because it was too correlated with **`Name_barton`**.
 - Dropped column **`Ticket_330909`** because it was too correlated with **`Name_sullivan`**.
 - Dropped column **`Ticket_330919`** because it was too correlated with **`Name_leary`**, **`Name_norah`**.
 - Dropped column **`Ticket_330923`** because it was too correlated with **`Name_mcgowan`**.
 - Dropped column **`Ticket_330931`** because it was too correlated with **`Name_mcgovern`**.
 - Dropped column **`Ticket_330932`** because it was too correlated with **`Name_brigdet`**, **`Name_mcdermott`**.
 - Dropped column **`Ticket_330935`** because it was too correlated with **`Name_peters`**.
 - Dropped column **`Ticket_330958`** because it was too correlated with **`Name_devaney`**.
 - Dropped column **`Ticket_330959`** because it was too correlated with **`Name_dwyer`**.
 - Dropped column **`Ticket_330980`** because it was too correlated with **`Name_ellie`**, **`Name_mockler`**.
 - Dropped column **`Ticket_33111`** because it was too correlated with **`Name_curnow`**, **`Name_jenkin`**.
 - Dropped column **`Ticket_3336`** because it was too correlated with **`Name_lobb`**.
 - Dropped column **`Ticket_3337`** because it was too correlated with **`Name_charity`**, **`Name_grace`**, **`Name_laury`**, **`Name_robins`**.
 - Dropped column **`Ticket_334912`** because it was too correlated with **`Name_connell`**.
 - Dropped column **`Ticket_335097`** because it was too correlated with **`Name_connaghton`**.
 - Dropped column **`Ticket_335677`** because it was too correlated with **`Name_agatha`**, **`Name_glynn`**.
 - Dropped column **`Ticket_33595`** because it was too correlated with **`Name_inglis`**, **`Name_milne`**, **`Name_watt`**.
 - Dropped column **`Ticket_33638`** because it was too correlated with **`Name_dodge`**.
 - Dropped column **`Ticket_3381`** because it was too correlated with **`Name_abelson`**.
 - Dropped column **`Ticket_34068`** because it was too correlated with **`Name_banfield`**.
 - Dropped column **`Ticket_3411`** because it was too correlated with **`Name_paulner`**, **`Name_uscher`**.
 - Dropped column **`Ticket_341826`** because it was too correlated with **`Name_adams`**.
 - Dropped column **`Ticket_34218`** because it was too correlated with **`Name_celia`**, **`Name_edwina`**, **`Name_troutt`**, **`Name_winnie`**.
 - Dropped column **`Ticket_34244`** because it was too correlated with **`Name_phillippe`**, **`Name_wiseman`**.
 - Dropped column **`Ticket_34260`** because it was too correlated with **`Name_lemore`**, **`Name_milley`**.
 - Dropped column **`Ticket_342826`** because it was too correlated with **`Name_sawyer`**.
 - Dropped column **`Ticket_343095`** because it was too correlated with **`Name_meek`**, **`Name_rowley`**.
 - Dropped column **`Ticket_343120`** because it was too correlated with **`Name_kristine`**, **`Name_salkjelsvik`**.
 - Dropped column **`Ticket_343275`** because it was too correlated with **`Name_celotti`**, **`Name_francesco`**.
 - Dropped column **`Ticket_343276`** because it was too correlated with **`Name_christmann`**.
 - Dropped column **`Ticket_345364`** because it was too correlated with **`Name_nysveen`**.
 - Dropped column **`Ticket_345572`** because it was too correlated with **`Name_guillaume`**, **`Name_messemaeker`**.
 - Dropped column **`Ticket_345763`** because it was too correlated with **`Name_emelia`**, **`Name_julius`**, **`Name_vandemoortele`**.
 - Dropped column **`Ticket_345765`** because it was too correlated with **`Name_cruyssen`**.
 - Dropped column **`Ticket_345767`** because it was too correlated with **`Name_achille`**, **`Name_waelens`**.
 - Dropped column **`Ticket_345769`** because it was too correlated with **`Name_hampe`**, **`Name_leon`**.
 - Dropped column **`Ticket_345770`** because it was too correlated with **`Name_cyriel`**, **`Name_nestor`**, **`Name_walle`**.
 - Dropped column **`Ticket_345773`** because it was too correlated with **`Name_impe`**.
 - Dropped column **`Ticket_345774`** because it was too correlated with **`Name_mulder`**, **`Name_theodore`**.
 - Dropped column **`Ticket_345777`** because it was too correlated with **`Name_melkebeke`**, **`Name_philemon`**.
 - Dropped column **`Ticket_345778`** because it was too correlated with **`Name_pelsmaeker`**.
 - Dropped column **`Ticket_345779`** because it was too correlated with **`Name_baptist`**, **`Name_jan`**, **`Name_sheerlinck`**.
 - Dropped column **`Ticket_345780`** because it was too correlated with **`Name_velde`**.
 - Dropped column **`Ticket_345781`** because it was too correlated with **`Name_aime`**, **`Name_lievens`**.
 - Dropped column **`Ticket_345783`** because it was too correlated with **`Name_steen`**, **`Name_vanden`**.
 - Dropped column **`Ticket_3464`** because it was too correlated with **`Name_crease`**.
 - Dropped column **`Ticket_34651`** because it was too correlated with **`Name_west`**.
 - Dropped column **`Ticket_347054`** because it was too correlated with **`Name_strom`**.
 - Dropped column **`Ticket_347061`** because it was too correlated with **`Name_ekstrom`**.
 - Dropped column **`Ticket_347062`** because it was too correlated with **`Name_joackim`**.
 - Dropped column **`Ticket_347064`** because it was too correlated with **`Name_widegren`**.
 - Dropped column **`Ticket_347067`** because it was too correlated with **`Name_bengt`**.
 - Dropped column **`Ticket_347069`** because it was too correlated with **`Name_gideon`**.
 - Dropped column **`Ticket_347071`** because it was too correlated with **`Name_agda`**, **`Name_lindahl`**, **`Name_thorilda`**, **`Name_viktoria`**.
 - Dropped column **`Ticket_347073`** because it was too correlated with **`Name_lindblom`**.
 - Dropped column **`Ticket_347074`** because it was too correlated with **`Name_eklund`**, **`Name_linus`**.
 - Dropped column **`Ticket_347076`** because it was too correlated with **`Name_petterson`**.
 - Dropped column **`Ticket_347077`** because it was too correlated with **`Name_asplund`**.
 - Dropped column **`Ticket_347078`** because it was too correlated with **`Name_fabian`**, **`Name_myhrman`**, **`Name_oliver`**, **`Name_pehr`**.
 - Dropped column **`Ticket_347080`** because it was too correlated with **`Name_danbom`**, **`Name_gilbert`**.
 - Dropped column **`Ticket_347081`** because it was too correlated with **`Name_nysten`**.
 - Dropped column **`Ticket_347083`** because it was too correlated with **`Name_ulrik`**.
 - Dropped column **`Ticket_347085`** because it was too correlated with **`Name_ohman`**, **`Name_velin`**.
 - Dropped column **`Ticket_347087`** because it was too correlated with **`Name_natalia`**, **`Name_pettersson`**.
 - Dropped column **`Ticket_347088`** because it was too correlated with **`Name_skoog`**.
 - Dropped column **`Ticket_347089`** because it was too correlated with **`Name_hedman`**.
 - Dropped column **`Ticket_347464`** because it was too correlated with **`Name_goransson`**.
 - Dropped column **`Ticket_347466`** because it was too correlated with **`Name_andreasson`**, **`Name_paul`**.
 - Dropped column **`Ticket_347468`** because it was too correlated with **`Name_augustsson`**.
 - Dropped column **`Ticket_347470`** because it was too correlated with **`Name_helmina`**, **`Name_josefina`**, **`Name_nilsson`**.
 - Dropped column **`Ticket_347743`** because it was too correlated with **`Name_lundahl`**.
 - Dropped column **`Ticket_348121`** because it was too correlated with **`Name_humblen`**, **`Name_nicolai`**.
 - Dropped column **`Ticket_348123`** because it was too correlated with **`Name_moen`**, **`Name_sigurd`**.
 - Dropped column **`Ticket_348124`** because it was too correlated with **`Name_soholt`**.
 - Dropped column **`Ticket_349201`** because it was too correlated with **`Name_ivanoff`**, **`Name_kanio`**.
 - Dropped column **`Ticket_349203`** because it was too correlated with **`Name_dantcheff`**, **`Name_ristiu`**.
 - Dropped column **`Ticket_349204`** because it was too correlated with **`Name_jonkoff`**.
 - Dropped column **`Ticket_349205`** because it was too correlated with **`Name_ilia`**, **`Name_stoytcheff`**.
 - Dropped column **`Ticket_349206`** because it was too correlated with **`Name_naidenoff`**, **`Name_penko`**.
 - Dropped column **`Ticket_349207`** because it was too correlated with **`Name_mionoff`**, **`Name_stoytcho`**.
 - Dropped column **`Ticket_349208`** because it was too correlated with **`Name_staneff`**.
 - Dropped column **`Ticket_349209`** because it was too correlated with **`Name_satio`**.
 - Dropped column **`Ticket_349210`** because it was too correlated with **`Name_peju`**.
 - Dropped column **`Ticket_349212`** because it was too correlated with **`Name_nedelio`**.
 - Dropped column **`Ticket_349213`** because it was too correlated with **`Name_marin`**, **`Name_markoff`**.
 - Dropped column **`Ticket_349214`** because it was too correlated with **`Name_petco`**, **`Name_slabenoff`**.
 - Dropped column **`Ticket_349215`** because it was too correlated with **`Name_pastcho`**, **`Name_pentcho`**.
 - Dropped column **`Ticket_349216`** because it was too correlated with **`Name_todoroff`**.
 - Dropped column **`Ticket_349217`** because it was too correlated with **`Name_kristo`**, **`Name_laleff`**.
 - Dropped column **`Ticket_349218`** because it was too correlated with **`Name_minko`**, **`Name_nankoff`**.
 - Dropped column **`Ticket_349219`** because it was too correlated with **`Name_danoff`**, **`Name_yoto`**.
 - Dropped column **`Ticket_349221`** because it was too correlated with **`Name_mitkoff`**, **`Name_mito`**.
 - Dropped column **`Ticket_349222`** because it was too correlated with **`Name_sdycoff`**, **`Name_todor`**.
 - Dropped column **`Ticket_349223`** because it was too correlated with **`Name_radeff`**.
 - Dropped column **`Ticket_349224`** because it was too correlated with **`Name_bostandyeff`**, **`Name_guentcho`**.
 - Dropped column **`Ticket_349225`** because it was too correlated with **`Name_denkoff`**, **`Name_mitto`**.
 - Dropped column **`Ticket_349227`** because it was too correlated with **`Name_plotcharsky`**, **`Name_vasil`**.
 - Dropped column **`Ticket_349228`** because it was too correlated with **`Name_branko`**, **`Name_dakic`**.
 - Dropped column **`Ticket_349231`** because it was too correlated with **`Name_cor`**, **`Name_liudevit`**.
 - Dropped column **`Ticket_349233`** because it was too correlated with **`Name_mineff`**.
 - Dropped column **`Ticket_349234`** because it was too correlated with **`Name_christo`**, **`Name_nenkoff`**.
 - Dropped column **`Ticket_349236`** because it was too correlated with **`Name_aloisia`**, **`Name_haas`**.
 - Dropped column **`Ticket_349237`** because it was too correlated with **`Name_franchi`**, **`Name_josef`**.
 - Dropped column **`Ticket_349239`** because it was too correlated with **`Name_stankovic`**.
 - Dropped column **`Ticket_349240`** because it was too correlated with **`Name_jalsevac`**.
 - Dropped column **`Ticket_349241`** because it was too correlated with **`Name_drazenoic`**, **`Name_jozef`**.
 - Dropped column **`Ticket_349242`** because it was too correlated with **`Name_pavlovic`**, **`Name_stefo`**.
 - Dropped column **`Ticket_349243`** because it was too correlated with **`Name_hendekovic`**, **`Name_ignjac`**.
 - Dropped column **`Ticket_349244`** because it was too correlated with **`Name_mara`**, **`Name_osman`**.
 - Dropped column **`Ticket_349245`** because it was too correlated with **`Name_petranec`**.
 - Dropped column **`Ticket_349246`** because it was too correlated with **`Name_karaic`**, **`Name_milan`**.
 - Dropped column **`Ticket_349247`** because it was too correlated with **`Name_stjepan`**, **`Name_turcin`**.
 - Dropped column **`Ticket_349248`** because it was too correlated with **`Name_balkic`**, **`Name_cerin`**.
 - Dropped column **`Ticket_349249`** because it was too correlated with **`Name_rekic`**, **`Name_tido`**.
 - Dropped column **`Ticket_349251`** because it was too correlated with **`Name_husein`**, **`Name_sivic`**.
 - Dropped column **`Ticket_349252`** because it was too correlated with **`Name_janko`**, **`Name_vovk`**.
 - Dropped column **`Ticket_349253`** because it was too correlated with **`Name_kraeff`**.
 - Dropped column **`Ticket_349254`** because it was too correlated with **`Name_gheorgheff`**, **`Name_stanio`**.
 - Dropped column **`Ticket_349256`** because it was too correlated with **`Name_karun`**, **`Name_manca`**.
 - Dropped column **`Ticket_349257`** because it was too correlated with **`Name_johann`**, **`Name_markun`**.
 - Dropped column **`Ticket_349909`** because it was too correlated with **`Name_palsson`**.
 - Dropped column **`Ticket_349910`** because it was too correlated with **`Name_lindell`**.
 - Dropped column **`Ticket_349912`** because it was too correlated with **`Name_edvardsson`**.
 - Dropped column **`Ticket_350025`** because it was too correlated with **`Name_juul`**.
 - Dropped column **`Ticket_350026`** because it was too correlated with **`Name_claus`**.
 - Dropped column **`Ticket_350029`** because it was too correlated with **`Name_damsgaard`**.
 - Dropped column **`Ticket_350034`** because it was too correlated with **`Name_jansson`**.
 - Dropped column **`Ticket_350036`** because it was too correlated with **`Name_eberhard`**, **`Name_thelander`**.
 - Dropped column **`Ticket_350042`** because it was too correlated with **`Name_sigfrid`**.
 - Dropped column **`Ticket_350043`** because it was too correlated with **`Name_wennerstrom`**.
 - Dropped column **`Ticket_350046`** because it was too correlated with **`Name_carla`**, **`Name_christine`**, **`Name_nielsine`**.
 - Dropped column **`Ticket_350047`** because it was too correlated with **`Name_niels`**.
 - Dropped column **`Ticket_350048`** because it was too correlated with **`Name_svend`**.
 - Dropped column **`Ticket_350404`** because it was too correlated with **`Name_albin`**, **`Name_klas`**, **`Name_klasen`**.
 - Dropped column **`Ticket_350406`** because it was too correlated with **`Name_adolfina`**, **`Name_amanda`**, **`Name_hulda`**, **`Name_vestrom`**.
 - Dropped column **`Ticket_350407`** because it was too correlated with **`Name_elina`**.
 - Dropped column **`Ticket_350417`** because it was too correlated with **`Name_jonsson`**.
 - Dropped column **`Ticket_35273`** because it was too correlated with **`Name_newell`**.
 - Dropped column **`Ticket_3536`** because it was too correlated with **`Name_cook`**.
 - Dropped column **`Ticket_3540`** because it was too correlated with **`Name_cohen`**, **`Name_gurshon`**, **`Name_gus`**.
 - Dropped column **`Ticket_35851`** because it was too correlated with **`Name_gilnagh`**.
 - Dropped column **`Ticket_35852`** because it was too correlated with **`Name_mullens`**.
 - Dropped column **`Ticket_3594`** because it was too correlated with **`Name_rouse`**.
 - Dropped column **`Ticket_36209`** because it was too correlated with **`Name_scanlan`**.
 - Dropped column **`Ticket_362316`** because it was too correlated with **`Name_reed`**.
 - Dropped column **`Ticket_363291`** because it was too correlated with **`Name_goldsmith`**.
 - Dropped column **`Ticket_363294`** because it was too correlated with **`Name_theobald`**.
 - Dropped column **`Ticket_364498`** because it was too correlated with **`Name_beard`**, **`Name_risien`**.
 - Dropped column **`Ticket_364499`** because it was too correlated with **`Name_portage`**, **`Name_tomlin`**.
 - Dropped column **`Ticket_364500`** because it was too correlated with **`Name_coxon`**.
 - Dropped column **`Ticket_364511`** because it was too correlated with **`Name_torber`**.
 - Dropped column **`Ticket_364512`** because it was too correlated with **`Name_brocklebank`**.
 - Dropped column **`Ticket_364846`** because it was too correlated with **`Name_canavan`**.
 - Dropped column **`Ticket_364850`** because it was too correlated with **`Name_mangan`**.
 - Dropped column **`Ticket_365222`** because it was too correlated with **`Name_burke`**, **`Name_jeremiah`**.
 - Dropped column **`Ticket_365226`** because it was too correlated with **`Name_hegarty`**.
 - Dropped column **`Ticket_36568`** because it was too correlated with **`Name_mcevoy`**.
 - Dropped column **`Ticket_367226`** because it was too correlated with **`Name_mccoy`**.
 - Dropped column **`Ticket_367228`** because it was too correlated with **`Name_mccormack`**.
 - Dropped column **`Ticket_367229`** because it was too correlated with **`Name_kiernan`**.
 - Dropped column **`Ticket_367230`** because it was too correlated with **`Name_murphy`**.
 - Dropped column **`Ticket_367231`** because it was too correlated with **`Name_carr`**.
 - Dropped column **`Ticket_367232`** because it was too correlated with **`Name_farrell`**.
 - Dropped column **`Ticket_367655`** because it was too correlated with **`Name_matthew`**, **`Name_sadlier`**.
 - Dropped column **`Ticket_36864`** because it was too correlated with **`Name_gallagher`**.
 - Dropped column **`Ticket_36865`** because it was too correlated with **`Name_kilgannon`**.
 - Dropped column **`Ticket_36866`** because it was too correlated with **`Name_mannion`**, **`Name_margareth`**.
 - Dropped column **`Ticket_368703`** because it was too correlated with **`Name_mernagh`**.
 - Dropped column **`Ticket_36928`** because it was too correlated with **`Name_wick`**.
 - Dropped column **`Ticket_36947`** because it was too correlated with **`Name_eustis`**.
 - Dropped column **`Ticket_36963`** because it was too correlated with **`Name_sutton`**.
 - Dropped column **`Ticket_36967`** because it was too correlated with **`Name_walker`**.
 - Dropped column **`Ticket_36973`** because it was too correlated with **`Name_birkhardt`**.
 - Dropped column **`Ticket_370129`** because it was too correlated with **`Name_rosblom`**.
 - Dropped column **`Ticket_370369`** because it was too correlated with **`Name_connors`**.
 - Dropped column **`Ticket_370370`** because it was too correlated with **`Name_madigan`**.
 - Dropped column **`Ticket_370371`** because it was too correlated with **`Name_lennon`**.
 - Dropped column **`Ticket_370372`** because it was too correlated with **`Name_mcmahon`**.
 - Dropped column **`Ticket_370373`** because it was too correlated with **`Name_connolly`**.
 - Dropped column **`Ticket_370375`** because it was too correlated with **`Name_healy`**.
 - Dropped column **`Ticket_370376`** because it was too correlated with **`Name_dooley`**.
 - Dropped column **`Ticket_370377`** because it was too correlated with **`Name_horgan`**.
 - Dropped column **`Ticket_371060`** because it was too correlated with **`Name_connor`**.
 - Dropped column **`Ticket_371362`** because it was too correlated with **`Name_cribb`**, **`Name_hatfield`**.
 - Dropped column **`Ticket_372622`** because it was too correlated with **`Name_morrow`**, **`Name_rowan`**.
 - Dropped column **`Ticket_374746`** because it was too correlated with **`Name_haim`**, **`Name_moutal`**, **`Name_rahamin`**.
 - Dropped column **`Ticket_374887`** because it was too correlated with **`Name_harmer`**, **`Name_lishin`**.
 - Dropped column **`Ticket_374910`** because it was too correlated with **`Name_shorney`**.
 - Dropped column **`Ticket_376564`** because it was too correlated with **`Name_thorneycroft`**.
 - Dropped column **`Ticket_376566`** because it was too correlated with **`Name_mcnamee`**.
 - Dropped column **`Ticket_37671`** because it was too correlated with **`Name_coutts`**.
 - Dropped column **`Ticket_382652`** because it was too correlated with **`Name_rice`**.
 - Dropped column **`Ticket_386525`** because it was too correlated with **`Name_davison`**, **`Name_finck`**.
 - Dropped column **`Ticket_3902`** because it was too correlated with **`Name_elsbury`**.
 - Dropped column **`Ticket_392076`** because it was too correlated with **`Name_sutehall`**.
 - Dropped column **`Ticket_392078`** because it was too correlated with **`Name_berk`**, **`Name_pickard`**, **`Name_trembisky`**.
 - Dropped column **`Ticket_392082`** because it was too correlated with **`Name_simmons`**.
 - Dropped column **`Ticket_392086`** because it was too correlated with **`Name_selman`**, **`Name_slocovski`**.
 - Dropped column **`Ticket_392087`** because it was too correlated with **`Name_meanwell`**, **`Name_ogden`**.
 - Dropped column **`Ticket_392089`** because it was too correlated with **`Name_sunderland`**.
 - Dropped column **`Ticket_392090`** because it was too correlated with **`Name_corn`**.
 - Dropped column **`Ticket_392091`** because it was too correlated with **`Name_aks`**, **`Name_leah`**, **`Name_rosen`**, **`Name_sam`**.
 - Dropped column **`Ticket_392092`** because it was too correlated with **`Name_sirota`**.
 - Dropped column **`Ticket_392096`** because it was too correlated with **`Name_moor`**.
 - Dropped column **`Ticket_39886`** because it was too correlated with **`Name_cater`**, **`Name_nosworthy`**.
 - Dropped column **`Ticket_4001`** because it was too correlated with **`Name_margido`**.
 - Dropped column **`Ticket_4133`** because it was too correlated with **`Name_lefebre`**.
 - Dropped column **`Ticket_4134`** because it was too correlated with **`Name_turkula`**.
 - Dropped column **`Ticket_4135`** because it was too correlated with **`Name_kristina`**, **`Name_laitinen`**.
 - Dropped column **`Ticket_4136`** because it was too correlated with **`Name_katriina`**.
 - Dropped column **`Ticket_4137`** because it was too correlated with **`Name_aina`**, **`Name_mari`**.
 - Dropped column **`Ticket_4138`** because it was too correlated with **`Name_turja`**.
 - Dropped column **`Ticket_4348`** because it was too correlated with **`Name_ivar`**, **`Name_sven`**.
 - Dropped column **`Ticket_45380`** because it was too correlated with **`Name_roland`**.
 - Dropped column **`Ticket_4579`** because it was too correlated with **`Name_siegwart`**.
 - Dropped column **`Ticket_541`** because it was too correlated with **`Name_jerwan`**, **`Name_marthe`**, **`Name_thuillard`**.
 - Dropped column **`Ticket_54510`** because it was too correlated with **`Name_moore`**.
 - Dropped column **`Ticket_5547`** because it was too correlated with **`Name_abbing`**.
 - Dropped column **`Ticket_5727`** because it was too correlated with **`Name_colley`**, **`Name_pomeroy`**.
 - Dropped column **`Ticket_5734`** because it was too correlated with **`Name_chaffee`**, **`Name_fuller`**, **`Name_herbert`**.
 - Dropped column **`Ticket_5735`** because it was too correlated with **`Name_crosby`**.
 - Dropped column **`Ticket_6212`** because it was too correlated with **`Name_shellard`**.
 - Dropped column **`Ticket_65303`** because it was too correlated with **`Name_ingvald`**, **`Name_olai`**.
 - Dropped column **`Ticket_65304`** because it was too correlated with **`Name_konrad`**, **`Name_reiersen`**.
 - Dropped column **`Ticket_65306`** because it was too correlated with **`Name_bernt`**, **`Name_bratthammer`**, **`Name_johannesen`**.
 - Dropped column **`Ticket_6563`** because it was too correlated with **`Name_olsvigen`**, **`Name_thor`**.
 - Dropped column **`Ticket_6607`** because it was too correlated with **`Name_johnston`**.
 - Dropped column **`Ticket_6608`** because it was too correlated with **`Name_ford`**.
 - Dropped column **`Ticket_6609`** because it was too correlated with **`Name_harknett`**, **`Name_phoebe`**.
 - Dropped column **`Ticket_693`** because it was too correlated with **`Name_nicholson`**.
 - Dropped column **`Ticket_695`** because it was too correlated with **`Name_frans`**.
 - Dropped column **`Ticket_7075`** because it was too correlated with **`Name_fredrik`**, **`Name_holm`**.
 - Dropped column **`Ticket_7076`** because it was too correlated with **`Name_adahl`**.
 - Dropped column **`Ticket_7077`** because it was too correlated with **`Name_adelia`**, **`Name_aurora`**, **`Name_landergren`**.
 - Dropped column **`Ticket_7267`** because it was too correlated with **`Name_odahl`**.
 - Dropped column **`Ticket_7546`** because it was too correlated with **`Name_ahlin`**, **`Name_persdotter`**.
 - Dropped column **`Ticket_7552`** because it was too correlated with **`Name_dahlberg`**, **`Name_gerda`**, **`Name_ulrika`**.
 - Dropped column **`Ticket_7553`** because it was too correlated with **`Name_strandberg`**.
 - Dropped column **`Ticket_7598`** because it was too correlated with **`Name_dahl`**, **`Name_edwart`**.
 - Dropped column **`Ticket_8471`** because it was too correlated with **`Name_danielsen`**, **`Name_gronnestad`**.
 - Dropped column **`Ticket_8475`** because it was too correlated with **`Name_halvorsen`**, **`Name_kalvik`**.
 - Dropped column **`Ticket_851`** because it was too correlated with **`Name_billiard`**, **`Name_blyler`**.
 - Dropped column **`Ticket_9549`** because it was too correlated with **`Name_sandstrom`**.
 - Dropped column **`Ticket_a4`** because it was too correlated with **`Name_moore`**, **`Ticket_54510`**.
 - Dropped column **`Ticket_basle`** because it was too correlated with **`Name_jerwan`**, **`Name_marthe`**, **`Name_thuillard`**, **`Ticket_541`**.
 - Dropped column **`Ticket_fa`** because it was too correlated with **`Name_ole`**, **`Ticket_265302`**.
 - Dropped column **`Ticket_sco`** because it was too correlated with **`Ticket_1585`**.
 - Dropped column **`Ticket_so`** because it was too correlated with **`Name_ilett`**, **`Ticket_14885`**.
 - Dropped column **`Ticket_sw`** because it was too correlated with **`Name_mellors`**.
 - Dropped column **`Ticket_we`** because it was too correlated with **`Name_crosby`**, **`Ticket_5735`**.
 - Dropped column **`Cabin_a10`** because it was too correlated with **`Name_ross`**, **`Ticket_13049`**.
 - Dropped column **`Cabin_a14`** because it was too correlated with **`Name_clifford`**, **`Name_quincy`**.
 - Dropped column **`Cabin_a16`** because it was too correlated with **`Name_christiana`**, **`Name_lady`**, **`Name_lucille`**, **`Name_sutherland`**, **`Ticket_11755`**.
 - Dropped column **`Cabin_a19`** because it was too correlated with **`Ticket_113056`**.
 - Dropped column **`Cabin_a20`** because it was too correlated with **`Name_cosmo`**, **`Name_edmund`**, **`Name_sir`**.
 - Dropped column **`Cabin_a23`** because it was too correlated with **`Name_algernon`**, **`Name_barkworth`**, **`Ticket_27042`**.
 - Dropped column **`Cabin_a24`** because it was too correlated with **`Name_roebling`**, **`Ticket_17590`**.
 - Dropped column **`Cabin_a26`** because it was too correlated with **`Name_blumer`**, **`Name_oberst`**, **`Name_simonius`**, **`Ticket_13213`**.
 - Dropped column **`Cabin_a31`** because it was too correlated with **`Name_blank`**, **`Ticket_112277`**.
 - Dropped column **`Cabin_a32`** because it was too correlated with **`Name_rood`**, **`Name_roscoe`**, **`Ticket_113767`**.
 - Dropped column **`Cabin_a34`** because it was too correlated with **`Name_dodge`**, **`Ticket_33638`**.
 - Dropped column **`Cabin_a36`** because it was too correlated with **`Ticket_112050`**.
 - Dropped column **`Cabin_a5`** because it was too correlated with **`Name_goldschmidt`**, **`Ticket_17754`**.
 - Dropped column **`Cabin_a6`** because it was too correlated with **`Name_sloper`**, **`Ticket_113788`**.
 - Dropped column **`Cabin_a7`** because it was too correlated with **`Name_clinch`**, **`Ticket_17764`**.
 - Dropped column **`Cabin_b101`** because it was too correlated with **`Name_gustave`**, **`Name_lesurer`**.
 - Dropped column **`Cabin_b102`** because it was too correlated with **`Name_fry`**, **`Ticket_112058`**.
 - Dropped column **`Cabin_b18`** because it was too correlated with **`Name_hippach`**, **`Ticket_111361`**.
 - Dropped column **`Cabin_b19`** because it was too correlated with **`Name_der`**, **`Name_hoef`**, **`Name_wyckoff`**, **`Ticket_111240`**.
 - Dropped column **`Cabin_b20`** because it was too correlated with **`Name_adrian`**, **`Name_dick`**.
 - Dropped column **`Cabin_b22`** because it was too correlated with **`Name_crosby`**, **`Ticket_5735`**, **`Ticket_we`**.
 - Dropped column **`Cabin_b28`** because it was too correlated with **`Embarked_nan`**, **`Ticket_113572`**.
 - Dropped column **`Cabin_b3`** because it was too correlated with **`Name_mcmillan`**, **`Name_scott`**.
 - Dropped column **`Cabin_b30`** because it was too correlated with **`Name_cornelius`**, **`Name_engelhart`**, **`Name_ostby`**, **`Ticket_113509`**.
 - Dropped column **`Cabin_b35`** because it was too correlated with **`Ticket_17477`**.
 - Dropped column **`Cabin_b37`** because it was too correlated with **`Name_kent`**, **`Ticket_11771`**.
 - Dropped column **`Cabin_b38`** because it was too correlated with **`Name_archibald`**, **`Name_butt`**, **`Name_willingham`**, **`Ticket_113050`**.
 - Dropped column **`Cabin_b39`** because it was too correlated with **`Name_margaritha`**, **`Ticket_13568`**.
 - Dropped column **`Cabin_b4`** because it was too correlated with **`Ticket_17610`**.
 - Dropped column **`Cabin_b41`** because it was too correlated with **`Name_maxmillian`**, **`Name_stehli`**, **`Ticket_13567`**.
 - Dropped column **`Cabin_b42`** because it was too correlated with **`Ticket_112053`**.
 - Dropped column **`Cabin_b49`** because it was too correlated with **`Name_bishop`**, **`Name_dickinson`**, **`Ticket_11967`**.
 - Dropped column **`Cabin_b50`** because it was too correlated with **`Name_maeglin`**, **`Name_max`**, **`Name_stahelin`**, **`Ticket_13214`**.
 - Dropped column **`Cabin_b53`** because it was too correlated with **`Cabin_b51`**.
 - Dropped column **`Cabin_b55`** because it was too correlated with **`Cabin_b51`**, **`Cabin_b53`**.
 - Dropped column **`Cabin_b57`** because it was too correlated with **`Name_ryerson`**, **`Ticket_17608`**.
 - Dropped column **`Cabin_b58`** because it was too correlated with **`Name_baxter`**, **`Ticket_17558`**.
 - Dropped column **`Cabin_b59`** because it was too correlated with **`Name_ryerson`**, **`Ticket_17608`**, **`Cabin_b57`**.
 - Dropped column **`Cabin_b60`** because it was too correlated with **`Name_baxter`**, **`Ticket_17558`**, **`Cabin_b58`**.
 - Dropped column **`Cabin_b63`** because it was too correlated with **`Name_ryerson`**, **`Ticket_17608`**, **`Cabin_b57`**, **`Cabin_b59`**.
 - Dropped column **`Cabin_b66`** because it was too correlated with **`Name_ryerson`**, **`Ticket_17608`**, **`Cabin_b57`**, **`Cabin_b59`**, **`Cabin_b63`**.
 - Dropped column **`Cabin_b69`** because it was too correlated with **`Name_gregg`**, **`Name_jennings`**, **`Name_melville`**.
 - Dropped column **`Cabin_b71`** because it was too correlated with **`Name_davidson`**, **`Ticket_12750`**.
 - Dropped column **`Cabin_b73`** because it was too correlated with **`Name_perreault`**.
 - Dropped column **`Cabin_b79`** because it was too correlated with **`Name_maioni`**, **`Name_roberta`**.
 - Dropped column **`Cabin_b80`** because it was too correlated with **`Name_elise`**, **`Name_lurette`**.
 - Dropped column **`Cabin_b82`** because it was too correlated with **`Name_guggenheim`**.
 - Dropped column **`Cabin_b84`** because it was too correlated with **`Name_guggenheim`**, **`Cabin_b82`**.
 - Dropped column **`Cabin_b86`** because it was too correlated with **`Name_giglio`**.
 - Dropped column **`Cabin_b94`** because it was too correlated with **`Name_harrison`**, **`Ticket_112059`**.
 - Dropped column **`Cabin_b96`** because it was too correlated with **`Ticket_113760`**.
 - Dropped column **`Cabin_b98`** because it was too correlated with **`Ticket_113760`**, **`Cabin_b96`**.
 - Dropped column **`Cabin_c101`** because it was too correlated with **`Name_appleton`**, **`Name_dale`**, **`Name_lamson`**, **`Ticket_11769`**.
 - Dropped column **`Cabin_c103`** because it was too correlated with **`Name_bonnell`**, **`Ticket_113783`**.
 - Dropped column **`Cabin_c104`** because it was too correlated with **`Name_peuchen`**, **`Ticket_113786`**.
 - Dropped column **`Cabin_c106`** because it was too correlated with **`Name_adolphe`**, **`Name_saalfeld`**, **`Ticket_19988`**.
 - Dropped column **`Cabin_c110`** because it was too correlated with **`Name_chamberlain`**, **`Name_porter`**.
 - Dropped column **`Cabin_c111`** because it was too correlated with **`Name_foreman`**, **`Name_laventall`**, **`Ticket_113051`**.
 - Dropped column **`Cabin_c118`** because it was too correlated with **`Name_natsch`**, **`Ticket_17596`**.
 - Dropped column **`Cabin_c123`** because it was too correlated with **`Name_futrelle`**, **`Name_heath`**, **`Ticket_113803`**.
 - Dropped column **`Cabin_c126`** because it was too correlated with **`Name_elmer`**, **`Name_taylor`**, **`Name_zebley`**, **`Ticket_19996`**.
 - Dropped column **`Cabin_c128`** because it was too correlated with **`Name_fellows`**, **`Name_fletcher`**, **`Name_lambert`**, **`Ticket_113510`**.
 - Dropped column **`Cabin_c148`** because it was too correlated with **`Name_behr`**, **`Name_howell`**, **`Ticket_111369`**.
 - Dropped column **`Cabin_c2`** because it was too correlated with **`Name_pears`**, **`Ticket_113776`**.
 - Dropped column **`Cabin_c22`** because it was too correlated with **`Name_allison`**.
 - Dropped column **`Cabin_c23`** because it was too correlated with **`Name_fortune`**, **`Ticket_19950`**.
 - Dropped column **`Cabin_c25`** because it was too correlated with **`Name_fortune`**, **`Ticket_19950`**, **`Cabin_c23`**.
 - Dropped column **`Cabin_c26`** because it was too correlated with **`Name_allison`**, **`Cabin_c22`**.
 - Dropped column **`Cabin_c27`** because it was too correlated with **`Name_fortune`**, **`Ticket_19950`**, **`Cabin_c23`**, **`Cabin_c25`**.
 - Dropped column **`Cabin_c30`** because it was too correlated with **`Name_markland`**, **`Name_molson`**, **`Ticket_113787`**.
 - Dropped column **`Cabin_c32`** because it was too correlated with **`Name_grice`**, **`Name_young`**.
 - Dropped column **`Cabin_c45`** because it was too correlated with **`Name_caroline`**, **`Name_endres`**.
 - Dropped column **`Cabin_c46`** because it was too correlated with **`Name_cavendish`**, **`Name_tyrell`**.
 - Dropped column **`Cabin_c47`** because it was too correlated with **`Name_marechal`**, **`Name_pierre`**, **`Ticket_11774`**.
 - Dropped column **`Cabin_c49`** because it was too correlated with **`Name_isham`**, **`Ticket_17595`**.
 - Dropped column **`Cabin_c50`** because it was too correlated with **`Name_alexenia`**, **`Name_potter`**.
 - Dropped column **`Cabin_c54`** because it was too correlated with **`Name_bechstein`**.
 - Dropped column **`Cabin_c62`** because it was too correlated with **`Name_astor`**, **`Name_force`**, **`Name_talmadge`**.
 - Dropped column **`Cabin_c64`** because it was too correlated with **`Name_astor`**, **`Name_force`**, **`Name_talmadge`**, **`Cabin_c62`**.
 - Dropped column **`Cabin_c65`** because it was too correlated with **`Name_castellana`**, **`Name_penasco`**, **`Name_satode`**, **`Ticket_17758`**.
 - Dropped column **`Cabin_c7`** because it was too correlated with **`Name_natalie`**.
 - Dropped column **`Cabin_c78`** because it was too correlated with **`Name_minahan`**, **`Ticket_19928`**.
 - Dropped column **`Cabin_c82`** because it was too correlated with **`Name_elkins`**, **`Name_widener`**, **`Ticket_113503`**.
 - Dropped column **`Cabin_c83`** because it was too correlated with **`Name_birkhardt`**, **`Ticket_36973`**.
 - Dropped column **`Cabin_c85`** because it was too correlated with **`Name_briggs`**, **`Name_cumings`**, **`Ticket_17599`**.
 - Dropped column **`Cabin_c86`** because it was too correlated with **`Name_donald`**.
 - Dropped column **`Cabin_c87`** because it was too correlated with **`Name_stead`**, **`Ticket_113514`**.
 - Dropped column **`Cabin_c90`** because it was too correlated with **`Name_antonine`**, **`Name_berthe`**, **`Name_mayne`**, **`Name_villiers`**, **`Ticket_17482`**.
 - Dropped column **`Cabin_c92`** because it was too correlated with **`Name_goldenberg`**, **`Ticket_17453`**.
 - Dropped column **`Cabin_c93`** because it was too correlated with **`Name_maxfield`**, **`Ticket_19943`**.
 - Dropped column **`Cabin_c95`** because it was too correlated with **`Name_farthing`**, **`Ticket_17483`**.
 - Dropped column **`Cabin_c99`** because it was too correlated with **`Name_bissette`**.
 - Dropped column **`Cabin_d10`** because it was too correlated with **`Name_greenfield`**, **`Ticket_17759`**.
 - Dropped column **`Cabin_d11`** because it was too correlated with **`Name_hogeboom`**.
 - Dropped column **`Cabin_d12`** because it was too correlated with **`Name_greenfield`**, **`Ticket_17759`**, **`Cabin_d10`**.
 - Dropped column **`Cabin_d15`** because it was too correlated with **`Name_albina`**, **`Name_bazzani`**, **`Ticket_11813`**.
 - Dropped column **`Cabin_d19`** because it was too correlated with **`Name_edwin`**, **`Name_kimball`**, **`Ticket_11753`**.
 - Dropped column **`Cabin_d20`** because it was too correlated with **`Name_eustis`**, **`Ticket_36947`**.
 - Dropped column **`Cabin_d21`** because it was too correlated with **`Name_kenyon`**, **`Ticket_17464`**.
 - Dropped column **`Cabin_d26`** because it was too correlated with **`Ticket_35281`**.
 - Dropped column **`Cabin_d28`** because it was too correlated with **`Name_conover`**, **`Name_lines`**, **`Ticket_17592`**.
 - Dropped column **`Cabin_d30`** because it was too correlated with **`Name_marvin`**, **`Name_warner`**, **`Ticket_113773`**.
 - Dropped column **`Cabin_d33`** because it was too correlated with **`Name_sleeper`**.
 - Dropped column **`Cabin_d35`** because it was too correlated with **`Name_beckwith`**, **`Ticket_11751`**.
 - Dropped column **`Cabin_d37`** because it was too correlated with **`Name_atkinson`**, **`Name_manley`**, **`Name_warren`**, **`Ticket_110813`**.
 - Dropped column **`Cabin_d45`** because it was too correlated with **`Name_hawksford`**, **`Ticket_16988`**.
 - Dropped column **`Cabin_d46`** because it was too correlated with **`Name_walker`**, **`Ticket_36967`**.
 - Dropped column **`Cabin_d47`** because it was too correlated with **`Name_newsom`**, **`Ticket_11752`**.
 - Dropped column **`Cabin_d48`** because it was too correlated with **`Name_webster`**.
 - Dropped column **`Cabin_d49`** because it was too correlated with **`Name_hammad`**, **`Name_hassab`**.
 - Dropped column **`Cabin_d50`** because it was too correlated with **`Name_sutton`**, **`Ticket_36963`**.
 - Dropped column **`Cabin_d56`** because it was too correlated with **`Name_beesley`**, **`Ticket_248698`**.
 - Dropped column **`Cabin_d6`** because it was too correlated with **`Name_clyde`**, **`Name_long`**, **`Name_milton`**, **`Ticket_113501`**.
 - Dropped column **`Cabin_d7`** because it was too correlated with **`Name_kornelia`**, **`Name_theodosia`**.
 - Dropped column **`Cabin_d9`** because it was too correlated with **`Name_fiske`**, **`Name_longley`**.
 - Dropped column **`Cabin_e10`** because it was too correlated with **`Name_berk`**, **`Name_pickard`**, **`Name_trembisky`**, **`Ticket_392078`**.
 - Dropped column **`Cabin_e12`** because it was too correlated with **`Ticket_19952`**.
 - Dropped column **`Cabin_e121`** because it was too correlated with **`Name_moor`**, **`Ticket_392096`**.
 - Dropped column **`Cabin_e17`** because it was too correlated with **`Ticket_113055`**.
 - Dropped column **`Cabin_e31`** because it was too correlated with **`Name_chaffee`**, **`Name_fuller`**, **`Name_herbert`**, **`Ticket_5734`**.
 - Dropped column **`Cabin_e33`** because it was too correlated with **`Name_bowerman`**, **`Ticket_113505`**.
 - Dropped column **`Cabin_e34`** because it was too correlated with **`Name_corning`**, **`Name_margaretta`**, **`Name_oakley`**, **`Name_spedden`**.
 - Dropped column **`Cabin_e36`** because it was too correlated with **`Name_francatelli`**, **`Name_laura`**.
 - Dropped column **`Cabin_e38`** because it was too correlated with **`Name_millet`**, **`Ticket_13509`**.
 - Dropped column **`Cabin_e40`** because it was too correlated with **`Name_burns`**.
 - Dropped column **`Cabin_e44`** because it was too correlated with **`Name_baird`**, **`Name_silvey`**, **`Ticket_13507`**.
 - Dropped column **`Cabin_e46`** because it was too correlated with **`Name_mccarthy`**, **`Ticket_17463`**.
 - Dropped column **`Cabin_e49`** because it was too correlated with **`Name_compton`**, **`Name_rebecca`**, **`Name_sara`**, **`Ticket_17756`**.
 - Dropped column **`Cabin_e50`** because it was too correlated with **`Name_achilles`**, **`Name_harder`**, **`Ticket_11765`**.
 - Dropped column **`Cabin_e58`** because it was too correlated with **`Name_colley`**, **`Name_pomeroy`**, **`Ticket_5727`**.
 - Dropped column **`Cabin_e63`** because it was too correlated with **`Name_gee`**, **`Ticket_111320`**.
 - Dropped column **`Cabin_e68`** because it was too correlated with **`Name_ruth`**.
 - Dropped column **`Cabin_e77`** because it was too correlated with **`Name_mack`**.
 - Dropped column **`Cabin_e8`** because it was too correlated with **`Name_chambers`**, **`Ticket_113806`**.
 - Dropped column **`Cabin_f2`** because it was too correlated with **`Name_navratil`**, **`Ticket_230080`**.
 - Dropped column **`Cabin_f38`** because it was too correlated with **`Ticket_383121`**.
 - Dropped column **`Cabin_f4`** because it was too correlated with **`Name_becker`**, **`Ticket_230136`**.
 - Dropped column **`Cabin_g63`** because it was too correlated with **`Name_humblen`**, **`Name_nicolai`**, **`Ticket_348121`**.

            

In [ ]:
import pickle

o = 200 # offset
n = 68  # # of samples
labels = titanic.iloc[o:(o+n)]['label']
predict_df = titanic.iloc[o:(o+n)].drop('label', axis=1).copy()

# labels = labels.reset_index()
predict_df.reset_index(inplace=True, drop=True)

m = pickle.loads(pm)
sum([ r == labels[o+i] for i, r in enumerate(m.run(predict_df)) ]) / n

NameError: name 'pm' is not defined

In [11]:
final_boss_automed = AutoMed(max_workers=2)
final_boss_automed.default_pipeline()
final_boss_results = final_boss_automed.fit(titanic.drop('label', axis=1).copy(), titanic[['label']].copy())

[10:53:18] running step: MetaOrderedStep (steps=MetaStep,MetaExplorerStep)

           running step: MetaStep                                                                                  
           (steps=ActSplitDate,ActDropTextualColumn,ActTfIdf,ActOnehot,ActDropDateColumn,ActMeanColumn,ActDropNumer
           icalColumn)

[10:53:20] running step: MetaStep (steps=ActRemoveHighCorrelatedColumn)

[10:53:28] running step: MetaStep (steps=ActRandomOverSampling,ActMinMaxScaler)

           running step: MetaExplorerStep (steps=WrapGeneticGridSearch)

...

           Traceback (most recent call last):                                                                      
             File "c:\Users\0869778\dev\automl\.venv\Lib\site-packages\pandas\core\indexes\base.py", line 3790, in 
           get_loc                                                                                                 
               return self._engine.get_loc(casted_key)                                                             
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                             
             File "index.pyx", line 152, in pandas._libs.index.IndexEngine.get_loc                                 
             File "index.pyx", line 181, in pandas._libs.index.IndexEngine.get_loc                                 
             File "pandas\_libs\hashtable_class_helper.pxi", line 7080, in                                         
           pandas._libs.hashtable.PyObjectHashTable.get_item                                                       
             File "pandas\_libs\hashtable_class_helper.pxi", line 7088, in                                         
           pandas._libs.hashtable.PyObjectHashTable.get_item                                                       
           KeyError: 0                                                                                             
                                                                                                                   
           The above exception was the direct cause of the following exception:                                    
                                                                                                                   
           Traceback (most recent call last):                                                                      
             File "c:\Users\0869778\dev\automl\src\sandbox\..\automed\worker_manager.py", line 35, in run          
               return self.task(*self.args, **self.kwargs)                                                         
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                         
             File "c:\Users\0869778\dev\automl\src\sandbox\..\automed\step.py", line 587, in runner_wrapper        
               output = func(self, current_input, callback=callback)                                               
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                               
             File "c:\Users\0869778\dev\automl\src\sandbox\..\automed\wrapper\dataset\wrap_kfold.py", line 63, in  
           run                                                                                                     
               metrics.append(output.evaluate(test_ds, force=True))                                                
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                 
             File "c:\Users\0869778\dev\automl\src\sandbox\..\automed\output.py", line 218, in evaluate            
               result = dataset.compute_metric(self.pipeline, metric)                                              
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                              
             File "c:\Users\0869778\dev\automl\src\sandbox\..\automed\dataset.py", line 299, in compute_metric     
               return metric.compute(self.__y, y_pred)                                                             
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                             
             File "c:\Users\0869778\dev\automl\src\sandbox\..\automed\metrics\f1_score_metric.py", line 56, in     
           compute                                                                                                 
               return f1_score(y, y_pred, pos_label=y[0]

           running step: WrapKFold (step=ActXGBoost, folds=5, stratify=True)

           running k-folds: ActXGBoost             (max_depth=14, random_state=42, learning_rate=0.552995971948809,
           n_estimators=206)

<class 'pandas.core.frame.DataFrame'>

<class 'pandas.core.frame.DataFrame'>

In [ ]:
# sum([ len(r.model.pickle()) for r in final_boss_results ])
[ (r.model.ml_model, r.evaluate()) for r in final_boss_results if r.model.ml_model is not None ]

NameError: name 'final_boss_results' is not defined